# Study 1 APS — trial-mapped EEG recovery (3 sets)

Converted from `study1_aps.py`. Run cells in order. The last cell starts the interactive scorer / plots.

Data live under `experiments/LevelN/Results_CSV` and `Results_xdf` (not in git). Outputs go to `analysis/outputs/`.


In [ ]:
# Run from repo root or from analysis/. This replaces __file__ used by the .py scripts.
from pathlib import Path
import os, sys
cwd = Path.cwd().resolve()
if (cwd / "analysis").is_dir() and (cwd / "experiments").is_dir():
    REPO_ROOT = cwd
elif cwd.name == "analysis":
    REPO_ROOT = cwd.parent
elif cwd.name == "notebooks":
    REPO_ROOT = cwd.parents[1]
else:
    REPO_ROOT = cwd
_ANALYSIS_DIR = REPO_ROOT / "analysis"
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print("REPO_ROOT:", REPO_ROOT)
%matplotlib inline


Unified EEG Analysis for Muse Headband Data - ALL LEVELS
V3: Trial-Aligned EEG Recovery (copy of eeg_analysis_all_levels.py)
====================================================================
KEY CHANGE FROM ORIGINAL:
  EEG recovery time uses TRIAL-ALIGNED BASELINE COMPARISON instead of
  the original local 1-second pre-stimulus window.

  Original (V1): Recovery = when post-stim EEG returns within ±10% of
                 the 1 second immediately before the stimulus.

  V3: For a distraction during trial N in Set 2/3:
       1. Find trial N's onset time in Set 1 behavioral data.
       2. Use the Set 1 EEG window starting at that onset time as the
          reference for what the brain looks like at that cognitive moment
          with NO distraction.
       3. Recovery = first sample post-distraction where the EEG matches
          the corresponding Set 1 reference sample within ±10%.

  WHY THIS IS BETTER THAN TIME-BASED MAPPING (V2):
    Distractions slow participants down — at t=45s in Set 2 a participant
    may be on Trial 5, while at t=45s in Set 1 they were already on Trial 9.
    Raw time-mapping compares incompatible cognitive states.
    Trial-based mapping aligns by *cognitive position* (same trial demand,
    same task moment), giving a physiologically meaningful reference:
    "What did the brain look like on Trial 5 when there was no distraction?"

  FALLBACK: If Set 1 behavioral or EEG data is unavailable, reverts to the
  original local pre-stim baseline method without crashing.
====================================================================

Includes Levels 1, 2, and 3 with both basic and advanced preprocessing options

LEVEL 1 Tasks:
- Classic (Go/No-Go)
- SART (Sustained Attention to Response Task)
- Semantic

LEVEL 2 Tasks:
- SelectiveVisual (Selective Visual Attention)
- ColorNumber (Color-Number Stroop)
- SpatialCue (Posner Spatial Cueing)

LEVEL 3 Tasks:
- PictureRecall (Picture Memory Recall)
- ColorLocation (Color-Location Binding)
- DualTask (Dual-Task)

Muse EEG channels:
- TP9: Left temporal
- AF7: Left frontal  
- AF8: Right frontal
- TP10: Right temporal


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from scipy.stats import pearsonr, zscore
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Try to import advanced preprocessing dependencies
try:
    from sklearn.decomposition import FastICA
    ADVANCED_AVAILABLE = True
except ImportError:
    ADVANCED_AVAILABLE = False
    print("Advanced preprocessing unavailable - install scikit-learn for ICA")

# GUI backend only outside notebooks
import sys
if "ipykernel" not in sys.modules:
try:
    matplotlib.use("TkAgg")
except Exception:
    pass

# Set style for better plots
try:
plt.style.use('seaborn-v0_8')
except:
plt.style.use('default')

sns.set_palette("husl")

class AllLevelsEEGAnalyzer:
"""Unified EEG analysis with all three levels and both basic and advanced preprocessing options"""

def __init__(self, base_path=None):
    repo = os.path.abspath(os.path.join(str(_ANALYSIS_DIR), ".."))
    if base_path is None:
        base_path = os.environ.get(
            "ATTENTIONSPAN_DATA", os.path.join(repo, "experiments")
        )
    self.base_path = base_path
    self.output_dir = os.environ.get(
        "ATTENTIONSPAN_OUTPUT", os.path.join(repo, "analysis", "outputs")
    )
    os.makedirs(self.output_dir, exist_ok=True)
    
    # Level-specific paths - All levels are in their own folders
    # Level 1: Inside Level1 folder
    self.level1_csv_path = os.path.join(base_path, "Level1/Results_CSV")
    self.level1_xdf_path = os.path.join(base_path, "Level1/Results_xdf")
    
    # Level 2: Inside Level2 folder
    self.level2_csv_path = os.path.join(base_path, "Level2/Results_CSV")
    self.level2_xdf_path = os.path.join(base_path, "Level2/Results_xdf")
    
    # Level 3: Inside Level3 folder
    self.level3_csv_path = os.path.join(base_path, "Level3/Results_CSV")
    self.level3_xdf_path = os.path.join(base_path, "Level3/Results_xdf")
    
    # Legacy paths for backward compatibility (old structure before level organization)
    self.xdf_path = os.path.join(base_path, "Results_xdf")
    self.csv_path = os.path.join(base_path, "Results_CSV")
    
    # Muse EEG channels
    self.eeg_channels = ['TP9', 'AF7', 'AF8', 'TP10']
    self.channel_locations = {
        'TP9': 'Left Temporal',
        'AF7': 'Left Frontal', 
        'AF8': 'Right Frontal',
        'TP10': 'Right Temporal'
    }
    
    # EEG frequency bands (Hz) - Standard definitions from MUSE website
    self.freq_bands = {
        'delta': (1, 4),      # Delta (δ) 1-4Hz
        'theta': (4, 8),      # Theta (θ) 4-8Hz  
        'alpha': (7.5, 13),   # Alpha (α) 7.5-13Hz
        'beta': (13, 30),     # Beta (β) 13-30Hz
        'gamma': (30, 44)     # Gamma (γ) 30-44Hz
    }
    
    # Level 1 Task mapping
    self.level1_task_names = {
        1: 'Classic',
        2: 'SART', 
        3: 'Semantic'
    }
    
    # Level 2 Task mapping
    self.level2_task_names = {
        1: 'selectiveVisual',
        2: 'colorNumber', 
        3: 'spatialCue'
    }
    
    # Level 3 Task mapping
    self.level3_task_names = {
        1: 'pictureRecall',
        2: 'colorLocation', 
        3: 'dualTask'
    }
    
    # Combined task mapping by level
    self.task_names_by_level = {
        1: self.level1_task_names,
        2: self.level2_task_names,
        3: self.level3_task_names
    }
    
    # Display names for better readability
    self.display_names = {
        # Level 1
        'Classic': 'Classic Go/No-Go',
        'SART': 'SART',
        'Semantic': 'Semantic',
        # Level 2
        'selectiveVisual': 'Selective Visual',
        'colorNumber': 'Color-Number',
        'spatialCue': 'Spatial Cue',
        # Level 3
        'pictureRecall': 'Picture Recall',
        'colorLocation': 'Color-Location',
        'dualTask': 'Dual Task'
    }
    
def get_task_name(self, level_id, task_id):
    """Get task name for a specific level and task ID"""
    if level_id in self.task_names_by_level:
        return self.task_names_by_level[level_id].get(task_id, f'Task{task_id}')
    return f'Level{level_id}_Task{task_id}'

def get_display_name(self, task_name):
    """Get human-readable display name for a task"""
    return self.display_names.get(task_name, task_name)

def get_accuracy_column(self, behavioral_data):
    """Get the correct accuracy column name from behavioral data"""
    if behavioral_data is None:
        return None
    for col in ['response_correct', 'is_correct', 'correct', 'accuracy']:
        if col in behavioral_data.columns:
            return col
    return None

def get_accuracy(self, behavioral_data):
    """Get accuracy value from behavioral data, handling different column names"""
    col = self.get_accuracy_column(behavioral_data)
    if col:
        return behavioral_data[col].mean()
    return np.nan

def get_csv_path_for_level(self, level_id, subject_id=None):
    """Get the CSV path for a specific level and optionally subject"""
    # All levels follow same structure: Level{N}/Results_CSV/Subject{M}/
    level_paths = {
        1: self.level1_csv_path,
        2: self.level2_csv_path,
        3: self.level3_csv_path
    }
    
    base = level_paths.get(level_id, self.csv_path)
    
    if subject_id is not None:
        # Try Subject folder first
        subject_path = os.path.join(base, f"Subject{subject_id}")
        if os.path.exists(subject_path):
            return subject_path
        
        # Try Level{N} subfolder (some legacy structures)
        level_subfolder = os.path.join(base, f"Level{level_id}")
        if os.path.exists(level_subfolder):
            return level_subfolder
        
        # Return subject path even if doesn't exist (will be created or error handled)
        return subject_path
    
    return base

def get_xdf_path_for_level(self, level_id, subject_id=None):
    """Get the XDF path for a specific level and optionally subject"""
    # All levels follow same structure: Level{N}/Results_xdf/Subject{M}/
    level_paths = {
        1: self.level1_xdf_path,
        2: self.level2_xdf_path,
        3: self.level3_xdf_path
    }
    
    base = level_paths.get(level_id, self.xdf_path)
    
    if subject_id is not None:
        # Try capital S first (Subject2), then lowercase (subject2) for legacy
        subject_path_cap = os.path.join(base, f"Subject{subject_id}")
        subject_path_low = os.path.join(base, f"subject{subject_id}")
        
        if os.path.exists(subject_path_cap):
            return subject_path_cap
        elif os.path.exists(subject_path_low):
            return subject_path_low
        # Default to capital S
        return subject_path_cap
    
    return base

def find_available_subjects(self, level_id):
    """Find available subjects for a given level"""
    subjects = set()
    
    csv_base = self.get_csv_path_for_level(level_id)
    xdf_base = self.get_xdf_path_for_level(level_id)
    
    # Check CSV directories
    if os.path.exists(csv_base):
        for item in os.listdir(csv_base):
            if item.startswith('Subject') and os.path.isdir(os.path.join(csv_base, item)):
                try:
                    subject_num = int(item.replace('Subject', ''))
                    subjects.add(subject_num)
                except ValueError:
                    pass
    
    # For Level 2 (flat structure), check file names
    if level_id == 2 and os.path.exists(csv_base):
        # Level 2 doesn't have subject folders, but we can still process
        subjects.add(0)  # Placeholder for flat structure
    
    # Check XDF directories (try both Subject and subject)
    if os.path.exists(xdf_base):
        for item in os.listdir(xdf_base):
            if item.startswith('Subject') and os.path.isdir(os.path.join(xdf_base, item)):
                try:
                    subject_num = int(item.replace('Subject', ''))
                    subjects.add(subject_num)
                except ValueError:
                    pass
            elif item.startswith('subject') and os.path.isdir(os.path.join(xdf_base, item)):
                try:
                    subject_num = int(item.replace('subject', ''))
                    subjects.add(subject_num)
                except ValueError:
                    pass
    
    return sorted(subjects)
    
def load_eeg_data(self, subject_id, level_id=1, task_id=None, set_id=None):
    """Load EEG data from XDF files (which may actually be CSV format with .xdf extension)"""
    
    # Get level-specific XDF path
    xdf_base = self.get_xdf_path_for_level(level_id, subject_id)
    
    # Try different file patterns
    file_patterns = []
    
    if task_id and set_id:
        # Pattern with curly braces (as seen in actual files)
        file_patterns.append(f"sub-{{{subject_id}}}_task{{{task_id}}}_set{{{set_id}}}.xdf")
        # Pattern without curly braces
        file_patterns.append(f"sub-{subject_id}_task{task_id}_set{set_id}.xdf")
        # Pattern with level
        file_patterns.append(f"sub-{{{subject_id}}}_level{{{level_id}}}_task{{{task_id}}}_set{{{set_id}}}.xdf")
    
    # Search directories (level-specific first)
    search_dirs = [
        xdf_base,
        os.path.join(self.base_path, f"Level{level_id}/Results_xdf/Subject{subject_id}"),
        os.path.join(self.base_path, f"Level{level_id}/Results_xdf/subject{subject_id}"),
    ]
    
    for search_dir in search_dirs:
        if search_dir is None or not os.path.exists(search_dir):
            continue
            
        for pattern in file_patterns:
            filepath = os.path.join(search_dir, pattern)
            if os.path.exists(filepath):
                print(f"Loading EEG data: {filepath}")
                
                # Try loading as CSV first (many .xdf files are actually CSV format)
                try:
                    data = pd.read_csv(filepath)
                    
                    # Check for EEG channels
                    available_channels = [ch for ch in self.eeg_channels if ch in data.columns]
                    if available_channels:
                        print(f"Found EEG channels (CSV format): {available_channels}")
                        if 'timestamps' in data.columns:
                            duration = data['timestamps'].iloc[-1] - data['timestamps'].iloc[0]
                            print(f"EEG samples: {len(data)}, Duration: {duration:.1f}s")
                        return data, filepath
                except Exception as csv_error:
                    pass  # Not CSV format, try XDF
                
                # Try loading with pyxdf (proper XDF format)
                try:
                    import pyxdf
                    streams, header = pyxdf.load_xdf(filepath)
                    
                    # Find EEG stream and convert to DataFrame
                    for stream in streams:
                        stream_type = stream['info']['type'][0] if 'type' in stream['info'] else ''
                        n_channels = int(stream['info']['channel_count'][0])
                        
                        if n_channels >= 4:  # Muse has 4 EEG channels
                            # Extract data
                            eeg_data = stream['time_series']
                            timestamps = stream['time_stamps']
                            
                            # Get channel names
                            ch_names = self.eeg_channels[:n_channels]
                            if 'desc' in stream['info'] and stream['info']['desc']:
                                try:
                                    channels = stream['info']['desc'][0]['channels'][0]['channel']
                                    ch_names = [ch['label'][0] for ch in channels]
                                except:
                                    pass
                            
                            # Create DataFrame
                            data = pd.DataFrame(eeg_data, columns=ch_names[:eeg_data.shape[1]])
                            data['timestamps'] = timestamps
                            
                            available_channels = [ch for ch in self.eeg_channels if ch in data.columns]
                            if available_channels:
                                print(f"Found EEG channels (XDF format): {available_channels}")
                                print(f"EEG samples: {len(data)}, Duration: {timestamps[-1] - timestamps[0]:.1f}s")
                                return data, filepath
                                
                except ImportError:
                    print("pyxdf not installed - trying CSV fallback")
                except Exception as e:
                    print(f"Error reading XDF format: {e}")
                    continue
    
    print(f"No valid EEG file found for subject {subject_id}, level {level_id}, task {task_id}, set {set_id}")
    return None, None

def load_behavioral_data(self, subject_id, task_name, set_id=None, level_id=1):
    """Load behavioral data from CSV files with level-specific paths"""
    
    # Get level-specific CSV path
    csv_dir = self.get_csv_path_for_level(level_id, subject_id)
    
    # For Level 2, also try the flat structure
    dirs_to_try = [csv_dir]
    if level_id == 2:
        dirs_to_try.append(self.level2_csv_path)
    elif level_id == 3:
        dirs_to_try.extend([
            os.path.join(self.level3_csv_path, f"Subject{subject_id}"),
            os.path.join(self.level3_csv_path, "Level3"),
            self.level3_csv_path
        ])
    
    for search_dir in dirs_to_try:
        if not os.path.exists(search_dir):
            continue
        
        try:
            csv_files = os.listdir(search_dir)
        except:
            continue
            
        # Try different patterns (including various case conventions)
        task_name_lower = task_name.lower()
        patterns = [
            f"{task_name}_results_set{set_id}_" if set_id else f"{task_name}_results_",
            f"{task_name}_results_",
            f"{task_name_lower}_results_set{set_id}_" if set_id else f"{task_name_lower}_results_",
            f"{task_name_lower}_results_"
        ]
        
        matching_files = []
        for pattern in patterns:
            matching = [f for f in csv_files if pattern in f and f.endswith('.csv')]
            if matching:
                matching_files.extend(matching)
                break
        
        if matching_files:
            # Use the most recent file
            csv_file = sorted(matching_files)[-1]
            filepath = os.path.join(search_dir, csv_file)
            print(f"Loading behavioral data: {filepath}")
            return pd.read_csv(filepath)
    
    print(f"No behavioral data found for Subject {subject_id}, Level {level_id}, Task {task_name}")
    return None

def preprocess_eeg(self, data, sampling_rate=256, advanced=False, **advanced_params):
    """Unified EEG preprocessing - basic or advanced"""
    
    if advanced and ADVANCED_AVAILABLE:
        return self._advanced_preprocessing(data, sampling_rate, **advanced_params)
    else:
        return self._basic_preprocessing(data, sampling_rate)

@staticmethod
def _fill_nans(series):
    """Robustly fill NaNs in a channel before it hits scipy.signal.filtfilt.

    pandas' default interpolate() only fills INTERIOR gaps — a NaN at the
    very first/last sample (headband settling, BT reconnect, etc.) survives.
    filtfilt has zero tolerance for NaNs: even one leftover NaN anywhere in
    the array poisons the ENTIRE filtered output (whole channel -> NaN),
    which then looks like a dead electrode / artifact and gets excluded
    from APS. limit_direction='both' fixes edge NaNs; the fillna(mean/0)
    is a belt-and-suspenders fallback for all-NaN or unreachable gaps.
    """
    cleaned = series.interpolate(method='linear', limit_direction='both')
    if cleaned.isna().any():
        fill_value = cleaned.mean()
        if pd.isna(fill_value):
            fill_value = 0.0
        cleaned = cleaned.fillna(fill_value)
    return cleaned

def _basic_preprocessing(self, data, sampling_rate=256):
    """Basic EEG preprocessing"""
    eeg_data = data[self.eeg_channels].copy()
    
    print("Preprocessing EEG data (basic)...")
    
    # 1. Remove obvious artifacts (values like -1000, 999)
    for channel in self.eeg_channels:
        if channel in eeg_data.columns:
            # Replace extreme values with NaN
            eeg_data.loc[eeg_data[channel] > 500, channel] = np.nan
            eeg_data.loc[eeg_data[channel] < -500, channel] = np.nan
            
            # Interpolate missing values (fills edges too, see _fill_nans)
            eeg_data[channel] = self._fill_nans(eeg_data[channel])
    
    # 2. Apply bandpass filter (1-44 Hz) to match frequency bands
    for channel in self.eeg_channels:
        if channel in eeg_data.columns and not eeg_data[channel].isna().all():
            # Design bandpass filter
            nyquist = sampling_rate / 2
            low = 1.0 / nyquist  # Match delta band lower limit
            high = 44.0 / nyquist  # Match gamma band upper limit
            
            b, a = signal.butter(4, [low, high], btype='band')
            eeg_data[channel] = signal.filtfilt(b, a, eeg_data[channel])
    
    print("Basic preprocessing completed")
    return eeg_data

def _advanced_preprocessing(self, data, sampling_rate=256, 
                          re_reference='average', notch_freq=60, apply_ica=True,
                          downsample_to=None, artifact_threshold=100):
    """Advanced EEG preprocessing pipeline"""
    
    print("Advanced EEG Preprocessing Pipeline")
    print("="*50)
    
    eeg_data = data[self.eeg_channels].copy()
    artifacts_removed = 0
    steps_applied = []
    
    # Step 1: Initial artifact removal
    print("Step 1: Initial artifact removal...")
    for channel in self.eeg_channels:
        if channel in eeg_data.columns:
            extreme_mask = (np.abs(eeg_data[channel]) > artifact_threshold)
            artifacts_removed += extreme_mask.sum()
            eeg_data.loc[extreme_mask, channel] = np.nan
            eeg_data[channel] = self._fill_nans(eeg_data[channel])
    steps_applied.append('extreme_artifact_removal')
    
    # Step 2: Notch filtering
    if notch_freq:
        print(f"Step 2: Notch filtering ({notch_freq} Hz line noise)...")
        nyquist = sampling_rate / 2
        freq = notch_freq / nyquist
        b, a = signal.iirnotch(freq, 30)  # Q factor = 30
        
        for channel in self.eeg_channels:
            if channel in eeg_data.columns:
                eeg_data[channel] = signal.filtfilt(b, a, eeg_data[channel])
        steps_applied.append(f'notch_filter_{notch_freq}Hz')
    
    # Step 3: Bandpass filtering
    print("Step 3: Bandpass filtering (1-44 Hz)...")
    nyquist = sampling_rate / 2
    low_norm = 1.0 / nyquist
    high_norm = 44.0 / nyquist
    b, a = signal.butter(4, [low_norm, high_norm], btype='band')
    
    for channel in self.eeg_channels:
        if channel in eeg_data.columns:
            eeg_data[channel] = signal.filtfilt(b, a, eeg_data[channel])
    steps_applied.append('bandpass_1-44Hz')
    
    # Step 4: Re-referencing
    if re_reference != 'none':
        print(f"Step 4: Re-referencing ({re_reference})...")
        if re_reference == 'average':
            avg_ref = eeg_data[self.eeg_channels].mean(axis=1)
            for channel in self.eeg_channels:
                if channel in eeg_data.columns:
                    eeg_data[channel] = eeg_data[channel] - avg_ref
        steps_applied.append(f'rereference_{re_reference}')
    
    # Step 5: ICA artifact removal
    if apply_ica and len(self.eeg_channels) >= 3:
        print("Step 5: ICA artifact removal...")
        try:
            # First, ensure all NaN and Inf values are cleaned in the DataFrame
            for channel in self.eeg_channels:
                if channel in eeg_data.columns:
                    # Replace Inf with NaN first
                    eeg_data[channel] = eeg_data[channel].replace([np.inf, -np.inf], np.nan)
                    
                    if eeg_data[channel].isna().any():
                        print(f"   → Cleaning NaN/Inf in {channel} before ICA...")
                        eeg_data[channel] = eeg_data[channel].interpolate(method='linear', limit_direction='both')
                        # Fill any remaining NaN with forward/backward fill
                        eeg_data[channel] = eeg_data[channel].ffill().bfill()
                        # If still NaN, use channel mean
                        if eeg_data[channel].isna().any():
                            channel_mean = eeg_data[channel].mean()
                            if np.isnan(channel_mean):
                                channel_mean = 0.0
                            eeg_data[channel] = eeg_data[channel].fillna(channel_mean)
            
            # Now extract clean data for ICA
            eeg_matrix = eeg_data[self.eeg_channels].values.T
            
            # Final cleanup: replace any remaining NaN or Inf with 0
            if np.any(np.isnan(eeg_matrix)) or np.any(np.isinf(eeg_matrix)):
                nan_count = np.sum(np.isnan(eeg_matrix))
                inf_count = np.sum(np.isinf(eeg_matrix))
                if nan_count > 0 or inf_count > 0:
                    print(f"   → Replacing {nan_count} NaN and {inf_count} Inf values...")
                eeg_matrix = np.nan_to_num(eeg_matrix, nan=0.0, posinf=0.0, neginf=0.0)
            
            # Run ICA
            print(f"   → Running ICA on {eeg_matrix.shape[0]} channels, {eeg_matrix.shape[1]} samples...")
            ica = FastICA(n_components=len(self.eeg_channels), random_state=42, max_iter=1000)
            sources = ica.fit_transform(eeg_matrix.T).T
            
            # Simple artifact detection (high variance components)
            artifact_components = []
            stds = [np.std(s) for s in sources]
            median_std = np.median(stds)
            
            for i, component in enumerate(sources):
                component_std = stds[i]
                if component_std > 3 * median_std:
                    artifact_components.append(i)
                    print(f"   → Component {i}: std={component_std:.2f} (artifact - likely blinks/jaw)")
            
            if artifact_components:
                print(f"   ✓ Removing {len(artifact_components)} artifact components")
                sources_clean = sources.copy()
                sources_clean[artifact_components, :] = 0
                eeg_clean = ica.inverse_transform(sources_clean.T).T
                
                for i, channel in enumerate(self.eeg_channels):
                    if channel in eeg_data.columns:
                        eeg_data[channel] = eeg_clean[i, :]
            else:
                print(f"   ✓ No obvious artifact components detected (data is clean)")
            
            steps_applied.append('ica_artifact_removal')
        except Exception as e:
            print(f"   ICA failed: {e}")
            print(f"   → Continuing without ICA artifact removal")
    
    # Step 6: Statistical artifact removal
    print("Step 6: Statistical artifact removal...")
    for channel in self.eeg_channels:
        if channel in eeg_data.columns:
            z_scores = np.abs(zscore(eeg_data[channel]))
            outlier_mask = z_scores > 4
            artifacts_removed += outlier_mask.sum()
            if outlier_mask.any():
                eeg_data.loc[outlier_mask, channel] = np.nan
                eeg_data[channel] = self._fill_nans(eeg_data[channel])
    steps_applied.append('statistical_artifact_removal')
    
    # Step 7: Downsampling (optional - disabled by default to preserve data)
    if downsample_to and downsample_to < sampling_rate:
        print(f"Step 7: Downsampling from {sampling_rate}Hz to {downsample_to}Hz...")
        decimation_factor = int(sampling_rate / downsample_to)
        
        print(f"   Original samples: {len(eeg_data)}")
        print(f"   Decimation factor: {decimation_factor}")
        
        # Anti-aliasing filter before downsampling
        nyquist = downsample_to / 2
        b, a = signal.butter(4, nyquist / (sampling_rate / 2), btype='low')
        
        downsampled_data = pd.DataFrame()
        for channel in self.eeg_channels:
            if channel in eeg_data.columns:
                # Apply anti-aliasing filter
                filtered = signal.filtfilt(b, a, eeg_data[channel])
                # Decimate
                downsampled_data[channel] = filtered[::decimation_factor]
        
        eeg_data = downsampled_data
        print(f"   Downsampled samples: {len(eeg_data)}")
        print(f"   Duration preserved: {len(eeg_data)/downsample_to:.1f}s")
        steps_applied.append(f'downsample_{downsample_to}Hz')
    else:
        print(f"Step 7: Skipping downsampling (keeping original {sampling_rate}Hz)")
    
    print("Advanced preprocessing completed!")
    print(f"Steps applied: {', '.join(steps_applied)}")
    print(f"Artifacts removed: {artifacts_removed}")
    
    return eeg_data

def extract_frequency_bands(self, eeg_data, sampling_rate=256):
    """Extract power in different frequency bands"""
    band_powers = {}
    
    for band_name, (low_freq, high_freq) in self.freq_bands.items():
        print(f"Extracting {band_name} band ({low_freq}-{high_freq} Hz)...")
        
        band_data = {}
        for channel in self.eeg_channels:
            if channel in eeg_data.columns:
                # Design bandpass filter for this frequency band
                nyquist = sampling_rate / 2
                low = low_freq / nyquist
                high = high_freq / nyquist
                
                channel_data = eeg_data[channel]
                if channel_data.isna().any():
                    # Belt-and-suspenders: a stray NaN here would poison
                    # filtfilt's output for the whole channel (see _fill_nans)
                    channel_data = self._fill_nans(channel_data)
                
                b, a = signal.butter(4, [low, high], btype='band')
                filtered = signal.filtfilt(b, a, channel_data)
                
                # Calculate power (RMS)
                power = np.sqrt(np.mean(filtered**2))
                band_data[channel] = power
        
        band_powers[band_name] = band_data
    
    return band_powers

def compute_trials_to_consecutive_correct(self, behavioral_data, num_consecutive=3):
    """
    Calculate trials needed to achieve N consecutive correct responses after each distraction.
    """
    if behavioral_data is None or 'stim_time' not in behavioral_data.columns:
        return {'mean': np.nan, 'per_stimulus': [], 'details': []}
    
    stim_indices = behavioral_data[behavioral_data['stim_time'].notna()].index.tolist()
    
    if len(stim_indices) == 0:
        return {'mean': np.nan, 'per_stimulus': [], 'details': []}
    
    trials_to_recovery = []
    recovery_details = []
    
    for stim_idx in stim_indices:
        # Look for consecutive correct responses AFTER this stimulus
        consecutive_count = 0
        trials_needed = 0
        
        for i in range(stim_idx + 1, len(behavioral_data)):
            trials_needed += 1
            
            # Check for correct response using available column
            acc_col = self.get_accuracy_column(behavioral_data)
            is_correct = behavioral_data.loc[i, acc_col] == 1 if acc_col else False
            
            if is_correct:
                consecutive_count += 1
                if consecutive_count >= num_consecutive:
                    # Found N consecutive correct responses
                    trials_to_recovery.append(trials_needed)
                    recovery_details.append({
                        'stim_trial': stim_idx,
                        'recovery_trial': i,
                        'trials_needed': trials_needed
                    })
                    break
            else:
                # Reset counter on incorrect response
                consecutive_count = 0
        else:
            # Did not achieve N consecutive correct before end
            trials_to_recovery.append(np.nan)
            recovery_details.append({
                'stim_trial': stim_idx,
                'recovery_trial': None,
                'trials_needed': np.nan
            })
    
    # Calculate mean (excluding NaN values)
    valid_recoveries = [t for t in trials_to_recovery if not np.isnan(t)]
    mean_trials = np.mean(valid_recoveries) if len(valid_recoveries) > 0 else np.nan
    
    return {
        'mean': mean_trials,
        'per_stimulus': trials_to_recovery,
        'details': recovery_details,
        'num_consecutive': num_consecutive
    }

def compute_behavioral_recovery(self, behavioral_data):
    """
    Compute behavioral recovery time.

    V3 change: recovery time is estimated as the sum of actual RTs across
    the recovery trials (from distraction to the first trial where RT returns
    within 10% of the pre-distraction baseline), rather than
    trials_to_recovery × baseline_RT.

    Using the real per-trial RTs gives the true elapsed time during the
    recovery window — if the participant was genuinely slower after
    distraction, that cost is captured rather than masked by baseline RT.
    """
    if behavioral_data is None or 'rt' not in behavioral_data.columns:
        return np.nan

    if 'stim_time' not in behavioral_data.columns:
        return np.nan

    distraction_indices = behavioral_data[behavioral_data['stim_time'].notna()].index.tolist()

    if len(distraction_indices) == 0:
        return np.nan

    # Smooth RT with a 3-trial rolling mean to reduce single-trial noise
    behavioral_data = behavioral_data.copy()
    behavioral_data['rt_smooth'] = behavioral_data['rt'].rolling(window=3, min_periods=1).mean()

    recovery_times = []

    for distraction_idx in distraction_indices:
        if distraction_idx >= len(behavioral_data) - 3:
            continue

        # Pre-distraction reference: mean of up to 3 trials before distraction
        baseline_start = max(0, distraction_idx - 3)
        baseline_rt = behavioral_data['rt_smooth'].iloc[baseline_start:distraction_idx].mean()

        if np.isnan(baseline_rt) or baseline_rt <= 0:
            continue

        # Walk forward from the distraction trial summing actual RTs
        # until RT returns within 10% of the pre-distraction baseline.
        # The sum of those real RTs is the true elapsed recovery time.
        elapsed_time = 0.0
        recovered = False
        for j in range(distraction_idx + 1, len(behavioral_data)):
            rt_j = behavioral_data['rt_smooth'].iloc[j]
            if np.isnan(rt_j):
                continue

            elapsed_time += behavioral_data['rt'].iloc[j]  # sum raw RT, not smoothed

            if abs(rt_j - baseline_rt) < 0.1 * baseline_rt:
                recovery_times.append(elapsed_time)
                print(f"   Behavioral recovery: {elapsed_time:.3f}s "
                      f"(sum of {j - distraction_idx} trial RTs, "
                      f"pre-distraction baseline RT={baseline_rt:.3f}s)")
                recovered = True
                break

        if not recovered:
            print(f"   Behavioral recovery: did not return to baseline RT within set")

    if len(recovery_times) == 0:
        return np.nan

    mean_behavioral_recovery = np.mean(recovery_times)
    print(f"Mean behavioral recovery time: {mean_behavioral_recovery:.3f}s")
    return mean_behavioral_recovery

def map_stim_times_to_baseline_trials(self, distracted_behavioral, baseline_behavioral):
    """
    Build a trial-aligned mapping: for each distraction stim_time in
    Set 2/3, find the EEG reference time in Set 1 by matching trial index.

    How it works
    ------------
    Both Set 1 and Set 2/3 run the exact same task, so row i in the
    behavioral CSV always corresponds to the same trial (e.g. row 4 is
    always the 5th stimulus shown).  The distraction column `stim_time`
    is non-NaN on the row where the distraction fired.

    For each such row at index i:
      - Locate row i in the Set 1 behavioral DataFrame.
      - Compute that row's time offset from the Set 1 task start using
        the `timestamp` column: t_ref = (trial_i_timestamp - first_timestamp)
      - t_ref is where we should look in Set 1 EEG to find the brain's
        undistracted state at that exact trial.

    Returns
    -------
    dict  {stim_time_in_set2 (float) → t_ref_in_set1 (float, seconds)}
          Empty dict if mapping cannot be built (missing columns etc.).
    """
    if distracted_behavioral is None or baseline_behavioral is None:
        return {}
    if 'stim_time' not in distracted_behavioral.columns:
        return {}
    if 'timestamp' not in baseline_behavioral.columns or 'timestamp' not in distracted_behavioral.columns:
        print("   [V3] Cannot build trial map — 'timestamp' column missing.")
        return {}

    try:
        set1_ts = pd.to_datetime(baseline_behavioral['timestamp'], errors='coerce')
        set1_start = set1_ts.iloc[0]
        # Offset of every Set 1 trial from Set 1 task start (seconds)
        set1_offsets = (set1_ts - set1_start).dt.total_seconds().values
    except Exception as e:
        print(f"   [V3] Failed to parse Set 1 timestamps: {e}")
        return {}

    mapping = {}
    distraction_rows = distracted_behavioral[distracted_behavioral['stim_time'].notna()]

    for iloc_pos, (df_idx, row) in enumerate(distraction_rows.iterrows()):
        stim_time = float(row['stim_time'])

        # Trial index = position of this row in the full distracted DataFrame
        trial_idx = distracted_behavioral.index.get_loc(df_idx)

        if trial_idx < len(set1_offsets):
            t_ref = float(set1_offsets[trial_idx])
            mapping[stim_time] = t_ref
            print(f"   [V3] stim@{stim_time:.2f}s (Set2 trial {trial_idx}) "
                  f"→ Set1 trial {trial_idx} at t={t_ref:.2f}s")
        else:
            # Trial doesn't exist in Set 1 — use last available Set 1 trial
            t_ref = float(set1_offsets[-1]) if len(set1_offsets) > 0 else stim_time
            mapping[stim_time] = t_ref
            print(f"   [V3] stim@{stim_time:.2f}s (Set2 trial {trial_idx}) "
                  f"→ beyond Set1 length, using last Set1 trial t={t_ref:.2f}s")

    return mapping

def compute_recovery_time(self, eeg_data, stim_times, baseline_window=1.0, recovery_window=3.0,
                          sampling_rate=256, baseline_eeg_data=None, trial_ref_map=None,
                          similarity_threshold=0.10):
    """
    Compute EEG recovery time per channel and averaged across channels.

    V3 METHOD (trial_ref_map + baseline_eeg_data provided):
    --------------------------------------------------------
    For each distraction stim_time:
      - Look up the corresponding Set 1 trial onset time via trial_ref_map.
      - Use the Set 1 EEG window [t_ref, t_ref + recovery_window] as the
        reference for the brain's undistracted state at that trial.
      - Recovery = first post-distraction sample within ±similarity_threshold
        of the corresponding Set 1 reference sample at that offset.

    Fallback (no trial_ref_map or no baseline_eeg_data):
    -----------------------------------------------------
    Uses original approach: 1-second local pre-stim baseline.
    """
    if len(stim_times) == 0 or eeg_data is None:
        return {ch: np.nan for ch in self.eeg_channels} | {'mean': np.nan}

    use_trial_map = (trial_ref_map is not None and len(trial_ref_map) > 0
                     and baseline_eeg_data is not None)
    method_label = "trial-aligned (V3)" if use_trial_map else "local pre-stim (fallback)"
    print(f"   EEG recovery method: {method_label}")

    recovery_times_per_channel = {}

    for channel in self.eeg_channels:
        if channel not in eeg_data.columns:
            recovery_times_per_channel[channel] = np.nan
            continue

        signal_data = eeg_data[channel].values
        ref_data = baseline_eeg_data[channel].values if (use_trial_map and channel in baseline_eeg_data.columns) else None
        recovery_times = []

        for stim_time in stim_times:
            stim_idx      = int(stim_time * sampling_rate)
            recovery_end  = min(int((stim_time + recovery_window) * sampling_rate), len(signal_data) - 1)

            if stim_idx >= len(signal_data):
                recovery_times.append(np.nan)
                continue

            post_stim = signal_data[stim_idx:recovery_end]
            if len(post_stim) == 0:
                recovery_times.append(np.nan)
                continue

            if use_trial_map and ref_data is not None:
                # ── TRIAL-ALIGNED METHOD (V3) ───────────────────────────
                # t_ref = Set 1 onset time of the same trial
                t_ref     = trial_ref_map.get(stim_time, None)
                if t_ref is None:
                    # No mapping for this stim_time — find closest key
                    closest  = min(trial_ref_map.keys(), key=lambda k: abs(k - stim_time))
                    t_ref    = trial_ref_map[closest]

                ref_start = int(t_ref * sampling_rate)
                ref_end   = min(int((t_ref + recovery_window) * sampling_rate), len(ref_data) - 1)

                if ref_start >= len(ref_data):
                    ref_segment = ref_data[-int(recovery_window * sampling_rate):]
                else:
                    ref_segment = ref_data[ref_start:ref_end]

                if len(ref_segment) == 0:
                    ref_segment = ref_data

                recovery_idx = None
                for i, val in enumerate(post_stim):
                    ref_val = ref_segment[min(i, len(ref_segment) - 1)]
                    epsilon = similarity_threshold * abs(ref_val) if abs(ref_val) > 1e-6 else similarity_threshold
                    if abs(val - ref_val) <= epsilon:
                        recovery_idx = i
                        break

                if recovery_idx is not None:
                    recovery_times.append(recovery_idx / sampling_rate)
                    print(f"   [{channel}] stim@{stim_time:.1f}s → recovered at "
                          f"+{recovery_idx/sampling_rate:.2f}s (Set1 trial ref t={t_ref:.2f}s)")
                else:
                    recovery_times.append(np.nan)
                    print(f"   [{channel}] stim@{stim_time:.1f}s → no recovery within "
                          f"{recovery_window}s (Set1 trial ref t={t_ref:.2f}s)")

            else:
                # ── ORIGINAL FALLBACK: local 1-s pre-stim baseline ──────
                baseline_start = int((stim_time - baseline_window) * sampling_rate)
                if baseline_start < 0:
                    recovery_times.append(np.nan)
                    continue
                baseline_power = np.mean(signal_data[baseline_start:stim_idx])
                epsilon = similarity_threshold * abs(baseline_power) if abs(baseline_power) > 1e-6 else similarity_threshold
                print(f"   [{channel}] stim@{stim_time:.1f}s local baseline: {baseline_power:.2f} μV ε={epsilon:.2f}")

                recovery_idx = None
                for i, val in enumerate(post_stim):
                    if abs(val - baseline_power) <= epsilon:
                        recovery_idx = i
                        break

                if recovery_idx is not None:
                    recovery_times.append(recovery_idx / sampling_rate)
                else:
                    recovery_times.append(np.nan)

        recovery_times_per_channel[channel] = np.nanmean(recovery_times)

    # Average across channels
    valid_vals = [v for v in recovery_times_per_channel.values() if not np.isnan(v)]
    recovery_times_per_channel['mean'] = np.nanmean(valid_vals) if valid_vals else np.nan

    print(f"   Mean EEG recovery (s): {recovery_times_per_channel.get('mean', np.nan):.3f} [{method_label}]")
    return recovery_times_per_channel

def compute_recovery_times_per_trial(self, eeg_signal, stim_times, baseline_window=1.0,
                                     recovery_window=3.0, sampling_rate=256, pct_of_baseline=0.1,
                                     baseline_eeg_signal=None, trial_ref_map=None):
    """
    Return per-trial recovery times (seconds).

    V3: If baseline_eeg_signal (Set 1 single-channel array) and trial_ref_map
    are provided, uses trial-aligned comparison — each post-stim sample is
    compared against the Set 1 signal at the same trial's onset time offset,
    rather than the local pre-stim window.
    Falls back to original local baseline method if not provided.
    """
    if len(stim_times) == 0:
        return np.array([])

    use_trial_map = (trial_ref_map is not None and len(trial_ref_map) > 0
                     and baseline_eeg_signal is not None)
    recovery_times = []

    for stim_time in stim_times:
        try:
            stim_idx     = int(stim_time * sampling_rate)
            recovery_end = min(int((stim_time + recovery_window) * sampling_rate),
                               len(eeg_signal) - 1)

            if stim_idx >= len(eeg_signal):
                recovery_times.append(np.nan)
                continue

            post_stim = eeg_signal[stim_idx:recovery_end]
            if len(post_stim) == 0:
                recovery_times.append(np.nan)
                continue

            if use_trial_map:
                # Trial-aligned: reference = Set 1 at same trial onset
                t_ref = trial_ref_map.get(stim_time, None)
                if t_ref is None:
                    closest = min(trial_ref_map.keys(), key=lambda k: abs(k - stim_time))
                    t_ref   = trial_ref_map[closest]

                ref_start = int(t_ref * sampling_rate)
                ref_end   = min(int((t_ref + recovery_window) * sampling_rate),
                                len(baseline_eeg_signal) - 1)

                if ref_start >= len(baseline_eeg_signal):
                    ref_seg = baseline_eeg_signal[-int(recovery_window * sampling_rate):]
                else:
                    ref_seg = baseline_eeg_signal[ref_start:ref_end]

                if len(ref_seg) == 0:
                    ref_seg = baseline_eeg_signal

                recovery_idx = None
                for i, val in enumerate(post_stim):
                    ref_val = ref_seg[min(i, len(ref_seg) - 1)]
                    epsilon = pct_of_baseline * abs(ref_val) if abs(ref_val) > 1e-6 else pct_of_baseline
                    if abs(val - ref_val) <= epsilon:
                        recovery_idx = i
                        break
            else:
                # Original: local 1-s pre-stim baseline
                baseline_start = int((stim_time - baseline_window) * sampling_rate)
                if baseline_start < 0:
                    recovery_times.append(np.nan)
                    continue
                baseline_power = np.mean(eeg_signal[baseline_start:stim_idx])
                epsilon = pct_of_baseline * abs(baseline_power) if abs(baseline_power) > 1e-6 else pct_of_baseline

                recovery_idx = None
                for i, value in enumerate(post_stim):
                    if abs(value - baseline_power) <= epsilon:
                        recovery_idx = i
                        break

            if recovery_idx is not None:
                recovery_times.append(recovery_idx / sampling_rate)
            else:
                recovery_times.append(np.nan)
        except Exception:
            recovery_times.append(np.nan)

    return np.array(recovery_times)

def compute_attention_span_score(self, behavioral_data, attention_metrics, recovery_time=None,
                                 eeg_metric_name='theta_beta_ratio', weights=None, per_trial=False,
                                 baseline_metrics=None, is_baseline=False):
    """
    Compute Attentional Resilience Score (ARS).
    
    Args:
        baseline_metrics: Dict with participant's baseline metrics from Set 1 (no distraction)
                         Keys: 'Accuracy', 'RT', 'EEG', 'TTGI'
        is_baseline: If True, this IS the baseline set - return 100 and raw metrics
    
    The ARS is normalized relative to the participant's own baseline:
    - Baseline (Set 1) always = 100 (reference point)
    - Sets 2 & 3 are scored relative to baseline performance
    - Recovery time is N/A for baseline (no distractions to recover from)
    """
    # Build metrics vector / DataFrame
    if per_trial:
        # Check for RT column
        rt_col = 'rt' if 'rt' in behavioral_data.columns else ('reaction_time' if 'reaction_time' in behavioral_data.columns else None)
        acc_col = self.get_accuracy_column(behavioral_data)
        
        if rt_col is None or acc_col is None:
            print(f"Cannot compute per-trial ARS: missing required trial columns (rt column: {rt_col}, accuracy column: {acc_col})")
            return None

        df = pd.DataFrame(index=behavioral_data.index)
        df['RT'] = behavioral_data[rt_col].astype(float)
        df['Accuracy'] = behavioral_data[acc_col].astype(float)
        df['TTGI'] = behavioral_data['time_to_give_in'] if 'time_to_give_in' in behavioral_data.columns else np.nan
        
        if (recovery_time is None) or (len(recovery_time) != len(df)):
            df['RecoveryLag'] = np.nan
        else:
            df['RecoveryLag'] = recovery_time
            
        if isinstance(attention_metrics.get(eeg_metric_name), dict):
            eeg_val = np.mean(list(attention_metrics[eeg_metric_name].values()))
            df['EEG'] = eeg_val
        else:
            df['EEG'] = attention_metrics.get(eeg_metric_name, np.nan)

        # Standardize per column (z-score) across trials
        z_df = df.apply(lambda col: (col - np.nanmean(col)) / (np.nanstd(col) if np.nanstd(col) > 0 else 1))
        z_df['RT'] = -z_df['RT']
        z_df['RecoveryLag'] = -z_df['RecoveryLag']
        if eeg_metric_name == 'theta_beta_ratio':
            z_df['EEG'] = -z_df['EEG']

        # Compose weights
        metric_cols = ['TTGI', 'Accuracy', 'RT', 'RecoveryLag', 'EEG']
        present = [c for c in metric_cols if c in z_df.columns]
        if weights is None:
            w = np.ones(len(present)) / len(present)
        else:
            if isinstance(weights, dict):
                w = np.array([weights.get(c, 0.0) for c in present], dtype=float)
            else:
                w = np.array(weights, dtype=float)
            w = w / np.sum(w)

        ass_series = z_df[present].fillna(0).values.dot(w)
        return pd.Series(ass_series, index=df.index, name='ARS')

    # ---------- Subject-level APS (7-component equal-weight formula) ----------
    # APS = 1/7 (Accuracy + RT + RT_SD + Recovery + ABR + TBR + FAA)
    # All components normalized relative to Set 1 baseline → baseline always = 100.
    # Scores above 100 are allowed when the participant outperforms baseline.
    # NaN recovery in distraction sets is penalized as 0 (did not recover).

    Accuracy = self.get_accuracy(behavioral_data)
    RT       = behavioral_data['rt'].mean() if 'rt' in behavioral_data.columns else (
               behavioral_data['reaction_time'].mean() if 'reaction_time' in behavioral_data.columns else np.nan)
    RT_std   = behavioral_data['rt'].std()  if 'rt' in behavioral_data.columns else np.nan
    RecoveryLag = (np.nan if is_baseline else
                   (recovery_time if np.isscalar(recovery_time) else
                    (np.nanmean(recovery_time) if recovery_time is not None else np.nan)))

    # Minimum meaningful EEG signal value — defined here so _mean_metric can use it.
    # A dead electrode produces ratio = 0/(0+1e-10) ≈ 0.0, not NaN.
    # Filtering below this threshold excludes dead channels from the per-channel average
    # so that one valid electrode is not dragged down by dead neighbours.
    EEG_MIN = 0.01

    # Extract all three EEG metrics (mean across valid channels only)
    def _mean_metric(key):
        val = attention_metrics.get(key, np.nan)
        if isinstance(val, dict):
            # Exclude NaN AND near-zero values — near-zero means dead electrode, not real signal
            vals = [v for v in val.values() if not np.isnan(v) and v > EEG_MIN]
            return float(np.mean(vals)) if vals else np.nan
        return float(val) if val is not None and not np.isnan(val) else np.nan

    TBR = _mean_metric('theta_beta_ratio')
    ABR = _mean_metric('alpha_beta_ratio')
    FAA = _mean_metric('frontal_asymmetry')

    # Full metrics dict — stored as baseline cache for Sets 2 & 3
    metrics_dict = {
        'Accuracy': Accuracy,
        'RT':       RT,
        'RT_std':   RT_std,
        'TBR':      TBR,
        'ABR':      ABR,
        'FAA':      FAA,
        # RecoveryLag not stored for baseline (no distractions), but included for sets 2/3
    }

    # ── BASELINE: always return 100 ──────────────────────────────────────
    if is_baseline:
        print("   BASELINE SET — APS = 100 (reference point)")
        print(f"   Cached: Accuracy={Accuracy:.1%}  RT={RT:.3f}s  RT_SD={RT_std:.3f}s"
              f"  TBR={TBR:.3f}  ABR={ABR:.3f}  FAA={FAA:.3f}")
        return 100.0, metrics_dict

    # ── SET 2/3: compute 7-component APS ─────────────────────────────────
    if baseline_metrics is None:
        print("   Warning: No baseline metrics available — APS cannot be normalized. Returning 0.")
        return 0.0

    component_scores = []
    component_labels = []
    skipped_labels   = []

    # EEG_MIN is defined above (before _mean_metric) and reused here for APS component guards.

    # Helper: log and append
    def _add(label, score):
        component_scores.append(score)
        component_labels.append(label)

    def _skip(label, reason):
        skipped_labels.append(label)
        print(f"   [{component_labels.__len__() + skipped_labels.__len__()}] "
              f"{label}: skipped — {reason}")

    # 1. Accuracy — higher is better → ratio = current / baseline
    b_acc = baseline_metrics.get('Accuracy', np.nan)
    if not np.isnan(Accuracy) and not np.isnan(b_acc) and b_acc > 0:
        s = Accuracy / b_acc
        _add('Accuracy', s)
        direction = f"+{(s-1)*100:.1f}%" if s >= 1 else f"{(s-1)*100:.1f}%"
        print(f"   [1] Accuracy:  {s:.3f}  ({Accuracy:.1%} vs baseline {b_acc:.1%}, {direction})")

    # 2. RT — lower is better → ratio = baseline / current
    b_rt = baseline_metrics.get('RT', np.nan)
    if not np.isnan(RT) and not np.isnan(b_rt) and b_rt > 0 and RT > 0:
        s = b_rt / RT
        _add('RT', s)
        direction = f"+{(s-1)*100:.1f}%" if s >= 1 else f"{(s-1)*100:.1f}%"
        print(f"   [2] RT:        {s:.3f}  ({RT:.3f}s vs baseline {b_rt:.3f}s, {direction})")

    # 3. RT_SD — lower is better (more consistent) → ratio = baseline_std / current_std
    b_rt_std = baseline_metrics.get('RT_std', np.nan)
    if not np.isnan(RT_std) and not np.isnan(b_rt_std) and b_rt_std > 0 and RT_std > 0:
        s = b_rt_std / RT_std
        _add('RT_SD', s)
        direction = f"+{(s-1)*100:.1f}% more consistent" if s >= 1 else f"{(s-1)*100:.1f}% less consistent"
        print(f"   [3] RT_SD:     {s:.3f}  ({RT_std:.3f}s vs baseline {b_rt_std:.3f}s, {direction})")

    # 4. Recovery Time — lower is better; NaN = 0 (penalty for non-recovery)
    # Baseline-relative scale: ceiling = RECOVERY_RT_MULTIPLIER x this participant's
    # OWN baseline RT (b_rt, computed above for the RT component) rather than one
    # universal constant shared by every participant. A naturally slow responder
    # (large b_rt) gets a proportionally longer allowed recovery window; a naturally
    # fast responder gets a tighter one — consistent with every other component
    # (Accuracy/RT/RT_SD/TBR/ABR/FAA) being scored against that same participant's
    # Set 1 values instead of a fixed threshold.
    RECOVERY_RT_MULTIPLIER = 10.0
    if np.isnan(RecoveryLag):
        recovery_score = 0.0
        _add('Recovery', recovery_score)
        print(f"   [4] Recovery:  0.000  (NaN — did not recover within window, penalized)")
    elif not np.isnan(b_rt) and b_rt > 0:
        recovery_ceiling_s = RECOVERY_RT_MULTIPLIER * b_rt
        recovery_score = max(0.0, 1.0 - RecoveryLag / recovery_ceiling_s)
        _add('Recovery', recovery_score)
        print(f"   [4] Recovery:  {recovery_score:.3f}  ({RecoveryLag:.3f}s vs "
              f"baseline-relative ceiling {recovery_ceiling_s:.2f}s = "
              f"{RECOVERY_RT_MULTIPLIER:.0f}x baseline RT {b_rt:.3f}s)")
    else:
        _skip('Recovery', f"baseline RT unavailable (b_rt={b_rt}) — cannot compute "
                           f"an individualized ceiling")

    # 5. TBR — lower is better (less mind-wandering) → ratio = baseline / current
    # If either value is near-zero the signal is an artifact — skip this component
    # and compute APS from the remaining valid ones.
    b_tbr = baseline_metrics.get('TBR', np.nan)
    if not np.isnan(TBR) and not np.isnan(b_tbr) and b_tbr > EEG_MIN and TBR > EEG_MIN:
        s = b_tbr / TBR
        _add('TBR', s)
        direction = f"+{(s-1)*100:.1f}% better focus" if s >= 1 else f"{(s-1)*100:.1f}% worse focus"
        print(f"   [5] TBR:       {s:.3f}  ({TBR:.3f} vs baseline {b_tbr:.3f}, {direction})")
    else:
        _skip('TBR', f"near-zero EEG signal (current={TBR:.4f}, baseline={b_tbr:.4f}) "
                     f"— likely artifact, excluded from APS")

    # 6. ABR — stability around baseline → score = 1/(1+|current−baseline|)
    # Score = 1.0 when identical, approaches 0 as deviation grows.
    # Both current and baseline must exceed EEG_MIN — if the alpha or beta
    # band was flat in either session the electrode contact was lost and the
    # value is an artifact, not a real neural signal.
    b_abr = baseline_metrics.get('ABR', np.nan)
    if (not np.isnan(ABR) and not np.isnan(b_abr)
            and abs(ABR) > EEG_MIN and abs(b_abr) > EEG_MIN):
        s = 1.0 / (1.0 + abs(ABR - b_abr))
        _add('ABR', s)
        print(f"   [6] ABR:       {s:.3f}  ({ABR:.3f} vs baseline {b_abr:.3f}, "
              f"deviation={abs(ABR-b_abr):.3f})")
    else:
        _skip('ABR', f"near-zero EEG signal (current={ABR:.4f}, baseline={b_abr:.4f}) "
                     f"— likely artifact, excluded from APS")

    # 7. FAA — stability around baseline → score = 1/(1+|current−baseline|)
    # FAA is now log-power and range-validated to [−1, +1] at computation time,
    # so NaN already means "invalid / out-of-range / dead electrode".
    # Both the current session and the baseline must be valid (non-NaN) for
    # normalisation to be meaningful.
    b_faa = baseline_metrics.get('FAA', np.nan)
    if not np.isnan(FAA) and not np.isnan(b_faa):
        s = 1.0 / (1.0 + abs(FAA - b_faa))
        _add('FAA', s)
        print(f"   [7] FAA:       {s:.3f}  (log-FAA {FAA:.3f} vs baseline {b_faa:.3f}, "
              f"deviation={abs(FAA-b_faa):.3f})")
    else:
        _skip('FAA', f"invalid log-FAA (current={FAA}, baseline={b_faa}) "
                     f"— out of [−1,+1] range or dead electrode, excluded from APS")

    if not component_scores:
        print("   Warning: No valid components — APS = 0")
        return (0.0, skipped_labels)

    # APS = equal-weight mean of valid component scores × 100
    # If any component was skipped (artifact/NaN), APS is calculated from the rest.
    aps_val = float(np.mean(component_scores)) * 100
    n       = len(component_scores)

    print(f"\n   APS = mean({n} components) × 100 = {aps_val:.2f}")
    print(f"   Components used ({n}/7): {component_labels}")
    if skipped_labels:
        print(f"   Skipped (artifact/missing data): {skipped_labels}")
    if aps_val > 100:
        print(f"   ✨ Above baseline — participant outperformed Set 1 on net")
    elif aps_val < 100:
        print(f"   ⚠️  Below baseline — distraction cost = {100-aps_val:.1f} APS points")

    # Return tuple (score, skipped_labels) so caller can record what was excluded
    return (aps_val, skipped_labels)


def calculate_attention_metrics(self, band_powers, eeg_data=None, behavioral_data=None,
                                baseline_eeg_data=None, trial_ref_map=None):
    """Calculate attention-related metrics including recovery time.

    baseline_eeg_data: Set 1 EEG DataFrame. Required for V3 trial-aligned recovery.
    trial_ref_map: dict {stim_time → t_ref_in_set1} from map_stim_times_to_baseline_trials().
    """
    metrics = {}
    
    # 1. Alpha/Beta ratio (common attention metric)
    if 'alpha' in band_powers and 'beta' in band_powers:
        alpha_beta_ratios = {}
        for channel in self.eeg_channels:
            if channel in band_powers['alpha'] and channel in band_powers['beta']:
                alpha_power = band_powers['alpha'][channel]
                beta_power = band_powers['beta'][channel]
                alpha_beta_ratios[channel] = alpha_power / (beta_power + 1e-10)
        metrics['alpha_beta_ratio'] = alpha_beta_ratios
    
    # 2. Theta/Beta ratio (attention vs. mind-wandering)
    if 'theta' in band_powers and 'beta' in band_powers:
        theta_beta_ratios = {}
        for channel in self.eeg_channels:
            if channel in band_powers['theta'] and channel in band_powers['beta']:
                theta_power = band_powers['theta'][channel]
                beta_power = band_powers['beta'][channel]
                theta_beta_ratios[channel] = theta_power / (beta_power + 1e-10)
        metrics['theta_beta_ratio'] = theta_beta_ratios
    
    # 3. Frontal asymmetry (AF7 vs AF8) — log-power FAA
    # Standard formula: FAA = ln(AF8_power) − ln(AF7_power)
    # where power = RMS²  (converting stored RMS back to mean square power).
    # Log transformation normalises for individual differences in overall alpha
    # amplitude and gives a dimensionless ratio that is comparable across
    # subjects.  The valid physiological range for healthy adults is −1 to +1
    # (a deviation of 1 means one hemisphere has ~2.72× the alpha power of the
    # other).  Values outside that window are treated as artifact and set to NaN.
    # The 1e-4 floor on the raw RMS prevents log(0) errors from dead electrodes.
    _FAA_RMS_MIN = 1e-4  # minimum RMS to keep log computation numerically safe
    if 'alpha' in band_powers:
        af7_rms = band_powers['alpha'].get('AF7', 0.0)
        af8_rms = band_powers['alpha'].get('AF8', 0.0)
        if af7_rms > _FAA_RMS_MIN and af8_rms > _FAA_RMS_MIN:
            # Convert RMS → mean-square power, then take log difference
            faa_val = np.log(af8_rms ** 2) - np.log(af7_rms ** 2)
            if -1.0 <= faa_val <= 1.0:
                metrics['frontal_asymmetry'] = faa_val
            else:
                metrics['frontal_asymmetry'] = np.nan
        else:
            metrics['frontal_asymmetry'] = np.nan
    
    # 4. EEG Recovery Time
    if eeg_data is not None and behavioral_data is not None:
        try:
            if 'stim_time' in behavioral_data.columns:
                stim_times = behavioral_data['stim_time'].values
                valid_stim_times = stim_times[~np.isnan(stim_times)]
                
                if len(valid_stim_times) > 0:
                    print(f"Found {len(valid_stim_times)} valid stimulus times for recovery analysis")
                    
                    # EEG Recovery — V3 trial-aligned if map available
                    eeg_recovery = self.compute_recovery_time(
                        eeg_data, valid_stim_times,
                        baseline_eeg_data=baseline_eeg_data,
                        trial_ref_map=trial_ref_map
                    )
                    eeg_mean = eeg_recovery['mean']
                
                    # Behavioral Recovery
                    behavioral_recovery = self.compute_behavioral_recovery(behavioral_data)
                    
                    # Trials to 3 consecutive correct responses
                    consecutive_correct_metric = self.compute_trials_to_consecutive_correct(behavioral_data, num_consecutive=3)
                    trials_to_recovery = consecutive_correct_metric['mean']

                    # Combined (equal weights)
                    if not np.isnan(eeg_mean) and not np.isnan(behavioral_recovery):
                        combined_recovery = 0.5 * eeg_mean + 0.5 * behavioral_recovery
                    elif not np.isnan(eeg_mean):
                        combined_recovery = eeg_mean
                    else:
                        combined_recovery = behavioral_recovery

                    # Store all recovery metrics
                    metrics['mean_recovery_time'] = combined_recovery
                    metrics['eeg_recovery_per_channel'] = eeg_recovery
                    metrics['behavioral_recovery'] = behavioral_recovery
                    metrics['trials_to_3_consecutive_correct'] = trials_to_recovery
                    metrics['consecutive_correct_details'] = consecutive_correct_metric['details']

                    print(f"Combined Recovery (EEG+Behavioral): {combined_recovery:.3f}s")
                    if not np.isnan(trials_to_recovery):
                        print(f"Trials to 3 Consecutive Correct: {trials_to_recovery:.1f} trials")
                    else:
                        print(f"Trials to 3 Consecutive Correct: Not achieved")
                    
                    # Plot recovery summary
                    try:
                        self.plot_recovery_summary(
                            eeg_recovery,
                            behavioral_recovery,
                            combined_recovery,
                            save_path=os.path.join(self.output_dir, "recovery_summary.png")
                        )
                    except Exception as e:
                        print(f"Could not create recovery plot: {e}")
                    
                    # Plot recovery time-series
                    try:
                        self.plot_recovery_timeseries(
                            eeg_data,
                            valid_stim_times,
                            channel='AF7',
                            baseline_window=1.0,
                            recovery_window=3.0,
                            sampling_rate=256,
                            save_path=os.path.join(self.output_dir, "recovery_timeseries.png")
                        )
                    except Exception as e:
                        print(f"Could not create recovery time-series plot: {e}")
                else:
                    print("No valid stimulus times found for recovery analysis")
                    metrics['mean_recovery_time'] = np.nan
                    metrics['eeg_recovery_per_channel'] = {}
                    metrics['behavioral_recovery'] = np.nan
            else:
                print("Missing stim_time column in behavioral data")
                metrics['mean_recovery_time'] = np.nan
                metrics['eeg_recovery_per_channel'] = {}
                metrics['behavioral_recovery'] = np.nan
                
        except Exception as e:
            print(f"Error computing combined recovery: {e}")
            metrics['mean_recovery_time'] = np.nan
            metrics['eeg_recovery_per_channel'] = {}
            metrics['behavioral_recovery'] = np.nan
    else:
        metrics['mean_recovery_time'] = np.nan
        metrics['eeg_recovery_per_channel'] = {}
        metrics['behavioral_recovery'] = np.nan
    
    return metrics

def print_analysis_summary(self, attention_metrics, behavioral_data, subject_id, level_id, task_id):
    """Print a clean, straightforward analysis summary"""
    task_name = self.get_task_name(level_id, task_id)
    display_name = self.get_display_name(task_name)
    
    print("\n" + "="*60)
    print(f"ATTENTION ANALYSIS REPORT - Subject {subject_id}")
    print(f"Level {level_id}, Task {task_id} ({display_name})")
    print("="*60)
    
    # Behavioral Performance
    if behavioral_data is not None:
        # Handle different column naming conventions
        if 'response_correct' in behavioral_data.columns:
            accuracy = behavioral_data['response_correct'].mean()
        elif 'is_correct' in behavioral_data.columns:
            accuracy = behavioral_data['is_correct'].mean()
        elif 'correct' in behavioral_data.columns:
            accuracy = behavioral_data['correct'].mean()
        else:
            accuracy = np.nan
        
        # Handle RT column
        if 'rt' in behavioral_data.columns:
            avg_rt = behavioral_data['rt'].mean()
            rt_std = behavioral_data['rt'].std()
        elif 'reaction_time' in behavioral_data.columns:
            avg_rt = behavioral_data['reaction_time'].mean()
            rt_std = behavioral_data['reaction_time'].std()
        else:
            avg_rt = np.nan
            rt_std = np.nan
        
        print(f"\nBEHAVIORAL PERFORMANCE:")
        print(f"   Accuracy:          {accuracy:.1%} ({accuracy*100:.1f}%)")
        print(f"   Average RT:        {avg_rt:.3f} seconds")
        print(f"   RT Variability:    {rt_std:.3f} seconds")
        
        # Performance interpretation
        if accuracy >= 0.90:
            accuracy_level = "Excellent"
        elif accuracy >= 0.80:
            accuracy_level = "Good"
        elif accuracy >= 0.70:
            accuracy_level = "Fair"
        else:
            accuracy_level = "Needs Improvement"
            
        print(f"   Performance Level: {accuracy_level}")
    
    # EEG Attention Metrics
    if attention_metrics:
        print(f"\nEEG ATTENTION METRICS:")
        
        # Alpha/Beta Ratio Analysis
        if 'alpha_beta_ratio' in attention_metrics:
            ab_ratio = np.mean(list(attention_metrics['alpha_beta_ratio'].values()))
            print(f"   Alpha/Beta Ratio:  {ab_ratio:.3f}")
            if ab_ratio < 0.5:
                ab_interpretation = "High focused attention"
            elif ab_ratio < 1.0:
                ab_interpretation = "Moderate attention"
            else:
                ab_interpretation = "Relaxed/internal focus"
            print(f"                      → {ab_interpretation}")
        
        # Theta/Beta Ratio Analysis
        if 'theta_beta_ratio' in attention_metrics:
            tb_ratio = np.mean(list(attention_metrics['theta_beta_ratio'].values()))
            print(f"   Theta/Beta Ratio:  {tb_ratio:.3f}")
            if tb_ratio < 0.5:
                tb_interpretation = "Very focused, minimal mind-wandering"
            elif tb_ratio < 1.0:
                tb_interpretation = "Good focus with some distractibility"
            else:
                tb_interpretation = "Higher mind-wandering tendency"
            print(f"                      → {tb_interpretation}")
        
        # Frontal Asymmetry Analysis
        if 'frontal_asymmetry' in attention_metrics:
            asymmetry = attention_metrics['frontal_asymmetry']
            print(f"   Frontal Asymmetry: {asymmetry:.3f}")
            if asymmetry > 0.1:
                asym_interpretation = "Right-hemisphere dominant (approach)"
            elif asymmetry < -0.1:
                asym_interpretation = "Left-hemisphere dominant (withdrawal)"
            else:
                asym_interpretation = "Balanced hemispheric activity"
            print(f"                      → {asym_interpretation}")
        
        # Recovery Time Analysis
        if 'mean_recovery_time' in attention_metrics:
            recovery_time = attention_metrics['mean_recovery_time']
            if not np.isnan(recovery_time):
                print(f"   Combined Recovery Time: {recovery_time:.3f}s")
                if recovery_time < 1.0:
                    recovery_interpretation = "Fast attentional recovery"
                elif recovery_time < 2.0:
                    recovery_interpretation = "Normal attentional recovery"
                else:
                    recovery_interpretation = "Slow attentional recovery"
                print(f"                          → {recovery_interpretation}")
                
                if 'behavioral_recovery' in attention_metrics and not np.isnan(attention_metrics['behavioral_recovery']):
                    print(f"     Behavioral Recovery: {attention_metrics['behavioral_recovery']:.3f}s")
                
                if 'eeg_recovery_per_channel' in attention_metrics:
                    eeg_rec = attention_metrics['eeg_recovery_per_channel']
                    if 'mean' in eeg_rec and not np.isnan(eeg_rec['mean']):
                        print(f"     EEG Recovery: {eeg_rec['mean']:.3f}s")
                
                if 'trials_to_3_consecutive_correct' in attention_metrics:
                    trials_to_recovery = attention_metrics['trials_to_3_consecutive_correct']
                    if not np.isnan(trials_to_recovery):
                        print(f"   Trials to 3 Consecutive Correct: {trials_to_recovery:.1f} trials")
                        if trials_to_recovery <= 3:
                            trials_interpretation = "Immediate recovery"
                        elif trials_to_recovery <= 5:
                            trials_interpretation = "Quick recovery"
                        elif trials_to_recovery <= 8:
                            trials_interpretation = "Moderate recovery"
                        else:
                            trials_interpretation = "Slow recovery"
                        print(f"                                  → {trials_interpretation}")
            else:
                print(f"   Recovery Time: Not available")
    
    # Overall Assessment
    print(f"\nOVERALL ASSESSMENT:")
    
    if behavioral_data is not None and attention_metrics:
        accuracy = self.get_accuracy(behavioral_data)
        
        if 'alpha_beta_ratio' in attention_metrics:
            ab_ratio = np.mean(list(attention_metrics['alpha_beta_ratio'].values()))
            
            if accuracy >= 0.90 and ab_ratio < 0.6:
                overall = "Excellent cognitive performance with strong attentional focus"
            elif accuracy >= 0.80 and ab_ratio < 0.8:
                overall = "Good cognitive performance with adequate attention"
            elif accuracy >= 0.70:
                overall = "Fair performance, possible attention fluctuations"
            else:
                overall = "Performance suggests attention difficulties"
            
            print(f"   {overall}")
    
    print("="*60)

def plot_recovery_summary(self, eeg_recovery, behavioral_recovery, combined_recovery, save_path=None):
    """Plot a comparison of EEG recovery per channel, behavioral recovery, and combined mean."""
    if eeg_recovery is None or len(eeg_recovery) == 0:
        print("No EEG recovery data available for plotting.")
        return

    channels = [ch for ch in eeg_recovery.keys() if ch != 'mean']
    eeg_values = [eeg_recovery.get(ch, np.nan) for ch in channels]
    
    valid_channels = []
    valid_eeg_values = []
    for ch, val in zip(channels, eeg_values):
        if not np.isnan(val):
            valid_channels.append(ch)
            valid_eeg_values.append(val)

    labels = valid_channels.copy()
    values = valid_eeg_values.copy()
    
    if not np.isnan(behavioral_recovery):
        labels.append("Behavioral")
        values.append(behavioral_recovery)
        
    if not np.isnan(combined_recovery):
        labels.append("Combined")
        values.append(combined_recovery)

    if len(values) == 0:
        print("No valid recovery data to plot.")
        return

    plt.figure(figsize=(10, 6))
    bars = plt.bar(labels, values, color=['lightblue', 'lightgreen', 'orange', 'red', 'purple', 'brown'][:len(values)])
    
    for bar, value in zip(bars, values):
        if not np.isnan(value):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                    f'{value:.3f}s', ha='center', va='bottom', fontsize=9)
    
    plt.ylabel("Recovery Time (seconds)")
    plt.title("EEG vs Behavioral vs Combined Recovery Time")
    plt.xticks(rotation=30)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Recovery plot saved to {save_path}")
        plt.close()
    else:
        plt.show()

def plot_recovery_timeseries(self, eeg_data, stim_times, channel='AF7', baseline_window=1.0, 
                            recovery_window=3.0, sampling_rate=256, save_path=None):
    """Plot time-series overlay showing how EEG returns to baseline around stimuli."""
    if channel not in eeg_data.columns:
        print(f"Channel {channel} not found in EEG data.")
        return
        
    valid_stim_times = stim_times[~np.isnan(stim_times)]
    
    if len(valid_stim_times) == 0:
        print("No valid stimulus times found for plotting.")
        return
        
    signal_data = eeg_data[channel].values
    
    baseline_samples = int(baseline_window * sampling_rate)
    recovery_samples = int(recovery_window * sampling_rate)
    total_samples = baseline_samples + recovery_samples
    time_axis = np.linspace(-baseline_window, recovery_window, total_samples)
    
    epochs = []
    recovery_points = []
    baselines = []
    
    for stim_time in valid_stim_times[:10]:
        stim_idx = int(stim_time * sampling_rate)
        epoch_start = stim_idx - baseline_samples
        epoch_end = stim_idx + recovery_samples
        
        if epoch_start < 0 or epoch_end >= len(signal_data):
            continue
            
        epoch = signal_data[epoch_start:epoch_end]
        epochs.append(epoch)
        
        baseline_power = np.mean(signal_data[epoch_start:stim_idx])
        baselines.append(baseline_power)
        
        epsilon = 0.1 * abs(baseline_power)
        post_stim_signal = signal_data[stim_idx:epoch_end]
        
        recovery_idx = None
        for i, val in enumerate(post_stim_signal):
            if abs(val - baseline_power) < epsilon:
                recovery_idx = i / sampling_rate
                break
        recovery_points.append(recovery_idx)
    
    if len(epochs) == 0:
        print("No valid epochs found for plotting.")
        return
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(epochs)))
    
    for i, (epoch, baseline, recovery_time, color) in enumerate(zip(epochs, baselines, recovery_points, colors)):
        if len(epoch) == len(time_axis):
            ax1.plot(time_axis, epoch, alpha=0.7, color=color, linewidth=1, 
                    label=f'Trial {i+1}' if i < 5 else "")
            
            ax1.axhline(y=baseline, color=color, linestyle='--', alpha=0.5, linewidth=1)
            
            epsilon = 0.1 * abs(baseline)
            ax1.axhline(y=baseline + epsilon, color=color, linestyle=':', alpha=0.3, linewidth=1)
            ax1.axhline(y=baseline - epsilon, color=color, linestyle=':', alpha=0.3, linewidth=1)
            
            if recovery_time is not None:
                recovery_amplitude = np.interp(recovery_time, time_axis, epoch)
                ax1.plot(recovery_time, recovery_amplitude, 'o', color=color, markersize=6, 
                        markeredgecolor='black', markeredgewidth=1)
    
    ax1.axvline(x=0, color='red', linestyle='-', alpha=0.8, linewidth=2, label='Stimulus Onset')
    ax1.set_xlabel('Time (seconds)')
    ax1.set_ylabel(f'{channel} Amplitude (μV)')
    ax1.set_title(f'EEG Recovery Time Series - {channel} Channel\n(Individual Trials with ε-based Detection)')
    ax1.grid(True, alpha=0.3)
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    epochs_array = np.array([epoch for epoch in epochs if len(epoch) == len(time_axis)])
    
    if len(epochs_array) > 0:
        mean_epoch = np.mean(epochs_array, axis=0)
        std_epoch = np.std(epochs_array, axis=0)
        
        ax2.plot(time_axis, mean_epoch, 'black', linewidth=2, label='Average Signal')
        ax2.fill_between(time_axis, mean_epoch - std_epoch, mean_epoch + std_epoch, 
                       alpha=0.3, color='gray', label='±1 SD')
        
        mean_baseline = np.mean(baselines)
        mean_epsilon = 0.1 * abs(mean_baseline)
        
        ax2.axhline(y=mean_baseline, color='blue', linestyle='--', linewidth=2, 
                   label=f'Baseline ({mean_baseline:.1f} μV)')
        ax2.axhline(y=mean_baseline + mean_epsilon, color='green', linestyle=':', linewidth=2,
                   label=f'ε bounds (±{mean_epsilon:.1f} μV)')
        ax2.axhline(y=mean_baseline - mean_epsilon, color='green', linestyle=':', linewidth=2)
        
        valid_recovery_times = [rt for rt in recovery_points if rt is not None]
        if valid_recovery_times:
            mean_recovery_time = np.mean(valid_recovery_times)
            ax2.axvline(x=mean_recovery_time, color='orange', linestyle='-', linewidth=2,
                       label=f'Avg Recovery ({mean_recovery_time:.2f}s)')
    
    ax2.axvline(x=0, color='red', linestyle='-', alpha=0.8, linewidth=2, label='Stimulus Onset')
    ax2.set_xlabel('Time (seconds)')
    ax2.set_ylabel(f'{channel} Amplitude (μV)')
    ax2.set_title(f'Average EEG Response with Recovery Detection\n({len(epochs)} trials averaged)')
    ax2.grid(True, alpha=0.3)
    ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()
    
    valid_recoveries = len([rt for rt in recovery_points if rt is not None])
    fig.suptitle(f'EEG Recovery Analysis: {valid_recoveries}/{len(epochs)} trials showed recovery within {recovery_window}s', 
                y=0.98, fontsize=14)
    
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Recovery time-series plot saved to {save_path}")
        plt.close()
    else:
        plt.show()

def visualize_results(self, eeg_data, band_powers, attention_metrics, behavioral_data=None, level_id=1, task_id=1):
    """Create comprehensive visualizations"""
    try:
        import matplotlib
        matplotlib.use('Agg')
        
        plt.ioff()
        fig = plt.figure(figsize=(16, 12))
        
        task_name = self.get_task_name(level_id, task_id)
        display_name = self.get_display_name(task_name)
        fig.suptitle(f'EEG Analysis - Level {level_id}: {display_name}', fontsize=14)
        
        # 1. Raw EEG time series
        plt.subplot(3, 3, 1)
        time_points = np.arange(len(eeg_data)) / 256
        max_points = min(5000, len(eeg_data))
        
        for i, channel in enumerate(self.eeg_channels):
            if channel in eeg_data.columns:
                data_subset = eeg_data[channel].iloc[:max_points].values
                time_subset = time_points[:max_points]
                plt.plot(time_subset, data_subset + i*100, 
                        label=f"{channel} ({self.channel_locations[channel]})")
        
        plt.title("Raw EEG Signals")
        plt.xlabel("Time (s)")
        plt.ylabel("Amplitude (μV)")
        plt.legend(fontsize=8)
        
        # 2. Frequency band powers by channel
        if band_powers:
            plt.subplot(3, 3, 2)
            channels = list(band_powers[list(band_powers.keys())[0]].keys())
            bands = list(band_powers.keys())
            
            x = np.arange(len(channels))
            width = 0.15
            
            for i, band in enumerate(bands):
                powers = [float(band_powers[band].get(ch, 0)) for ch in channels]
                plt.bar(x + i*width, powers, width, label=band)
            
            plt.title("Frequency Band Powers by Channel")
            plt.xlabel("Channels")
            plt.ylabel("Power")
            plt.xticks(x + width*2, channels)
            plt.legend(fontsize=8)
        
        # 3. Alpha/Beta ratios
        if 'alpha_beta_ratio' in attention_metrics:
            plt.subplot(3, 3, 3)
            ratios = attention_metrics['alpha_beta_ratio']
            channels = list(ratios.keys())
            values = [float(v) for v in ratios.values()]
            
            plt.bar(channels, values, color='skyblue')
            plt.title("Alpha/Beta Ratio (Attention Metric)")
            plt.ylabel("Ratio")
            plt.xticks(rotation=45)
        
        # 4. Theta/Beta ratios
        if 'theta_beta_ratio' in attention_metrics:
            plt.subplot(3, 3, 4)
            ratios = attention_metrics['theta_beta_ratio']
            channels = list(ratios.keys())
            values = [float(v) for v in ratios.values()]
            
            plt.bar(channels, values, color='lightcoral')
            plt.title("Theta/Beta Ratio (Mind-wandering Metric)")
            plt.ylabel("Ratio")
            plt.xticks(rotation=45)
        
        # 5. Frequency band comparison
        if band_powers:
            plt.subplot(3, 3, 5)
            avg_powers = {}
            for band, powers in band_powers.items():
                avg_powers[band] = float(np.mean(list(powers.values())))
            
            plt.bar(list(avg_powers.keys()), list(avg_powers.values()), color='lightgreen')
            plt.title("Average Power Across All Channels")
            plt.ylabel("Power")
            plt.xticks(rotation=45)
        
        # 6. Behavioral data (if available)
        if behavioral_data is not None:
            plt.subplot(3, 3, 6)
            rt_values = behavioral_data['rt'].values
            plt.hist(rt_values, bins=30, alpha=0.7, color='orange')
            plt.title("Reaction Time Distribution")
            plt.xlabel("Reaction Time (s)")
            plt.ylabel("Count")
            
            plt.subplot(3, 3, 7)
            acc_col = self.get_accuracy_column(behavioral_data)
            if acc_col:
                accuracy_over_time = behavioral_data[acc_col].rolling(window=10).mean()
                plt.plot(accuracy_over_time.values, color='purple')
                plt.title("Accuracy Over Time (10-trial moving average)")
                plt.xlabel("Trial Number")
                plt.ylabel("Accuracy")
            else:
                plt.text(0.5, 0.5, 'No accuracy data', ha='center', va='center')
            
            plt.subplot(3, 3, 8)
            rt_over_time = behavioral_data['rt'].rolling(window=10).mean()
            plt.plot(rt_over_time.values, color='brown')
            plt.title("Reaction Time Over Time (10-trial moving average)")
            plt.xlabel("Trial Number")
            plt.ylabel("RT (s)")
        
        # 9. Frontal asymmetry (if available)
        if 'frontal_asymmetry' in attention_metrics:
            plt.subplot(3, 3, 9)
            asymmetry = float(attention_metrics['frontal_asymmetry'])
            plt.bar(['Frontal Asymmetry'], [asymmetry], 
                   color='red' if asymmetry > 0 else 'blue')
            plt.title("Frontal Alpha Asymmetry")
            plt.ylabel("AF8 - AF7 (μV)")
            plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)
        
        plt.tight_layout()
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        save_path = os.path.join(self.base_path, f"eeg_analysis_L{level_id}T{task_id}_{timestamp}.png")
        
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        
        plt.savefig(save_path, dpi=300, bbox_inches='tight', 
                   facecolor='white', edgecolor='none')
        print(f"Plot saved successfully as: {save_path}")
        
        plt.close(fig)
        
    except Exception as e:
        print(f"Visualization error: {e}")
        print("The analysis completed successfully, but there was an issue with the plot display.")

def analyze_subject(self, subject_id, level_id=1, task_id=1, set_id=1, advanced_preprocessing=False,
                    baseline_metrics=None, baseline_eeg_data=None, baseline_behavioral_data=None,
                    **advanced_params):
    """
    Complete analysis for one subject with unified preprocessing options.

    Args:
        baseline_metrics: Dict with baseline metrics from Set 1 for this task.
        baseline_eeg_data: Preprocessed EEG DataFrame from Set 1. Used to build
                           the trial-aligned EEG reference for V3 recovery.
        baseline_behavioral_data: Behavioral DataFrame from Set 1. Used with
                           map_stim_times_to_baseline_trials() to create the
                           trial → t_ref mapping.

    Returns:
        Dict with results including 'baseline_metrics', 'eeg_data', and
        'behavioral_data' for Set 1 (all cached for Sets 2 & 3).
    """
    task_name = self.get_task_name(level_id, task_id)
    display_name = self.get_display_name(task_name)
    
    # Determine if this is a baseline set (Set 1 = no distractions)
    is_baseline = (set_id == 1)
    
    print(f"\n{'='*60}")
    preprocessing_type = "Advanced" if advanced_preprocessing else "Basic"
    print(f"ANALYZING: Subject {subject_id}")
    print(f"Level {level_id}, Task {task_id} ({display_name}), Set {set_id}")
    print(f"Preprocessing: {preprocessing_type}")
    if is_baseline:
        print("*** BASELINE SET (no distractions) ***")
    elif baseline_metrics:
        print("*** Using baseline-relative normalization ***")
    print(f"{'='*60}")
    
    # 1. Load EEG data
    eeg_raw, eeg_filepath = self.load_eeg_data(subject_id, level_id, task_id, set_id)
    if eeg_raw is None:
        return None
    
    # 2. Preprocess EEG (basic or advanced)
    eeg_data = self.preprocess_eeg(eeg_raw, advanced=advanced_preprocessing, **advanced_params)
    
    # 3. Extract frequency bands
    band_powers = self.extract_frequency_bands(eeg_data)
    
    # 4. Load behavioral data
    behavioral_data = self.load_behavioral_data(subject_id, task_name, set_id, level_id)

    # 5a. Build trial-aligned reference map for Sets 2 & 3 (V3)
    trial_ref_map = None
    if not is_baseline and behavioral_data is not None and baseline_behavioral_data is not None:
        print("   Building trial-aligned EEG reference map (V3)...")
        trial_ref_map = self.map_stim_times_to_baseline_trials(
            distracted_behavioral=behavioral_data,
            baseline_behavioral=baseline_behavioral_data
        )
        if trial_ref_map:
            print(f"   Trial map built: {len(trial_ref_map)} distraction(s) mapped to Set 1 trial offsets")
        else:
            print("   Trial map empty — will fall back to local pre-stim baseline")

    # 5b. Calculate attention metrics (including recovery time)
    attention_metrics = self.calculate_attention_metrics(
        band_powers, eeg_data, behavioral_data,
        baseline_eeg_data=baseline_eeg_data,
        trial_ref_map=trial_ref_map
    )
    
    # Extract RecoveryLag for attention span scoring (N/A for baseline)
    if is_baseline:
        RecoveryLag = np.nan  # No recovery to measure in baseline
    else:
        RecoveryLag = attention_metrics.get('mean_recovery_time', np.nan)

    # Compute per-trial recovery times (only for distraction sets)
    per_trial_recovery = None
    if not is_baseline and eeg_data is not None and behavioral_data is not None and 'stim_time' in behavioral_data.columns:
        stim_times = behavioral_data['stim_time'].values
        if 'AF7' in eeg_data.columns:
            af7_signal   = eeg_data['AF7'].values
            af7_baseline = (baseline_eeg_data['AF7'].values
                            if baseline_eeg_data is not None and 'AF7' in baseline_eeg_data.columns
                            else None)
            per_trial_recovery = self.compute_recovery_times_per_trial(
                af7_signal, stim_times, baseline_window=1.0, recovery_window=3.0, sampling_rate=256,
                baseline_eeg_signal=af7_baseline,
                trial_ref_map=trial_ref_map
            )

    # Compute ARS (subject-level)
    print("\n" + "="*60)
    print("ATTENTIONAL RESILIENCE SCORE (ARS) CALCULATION")
    if is_baseline:
        print("(Baseline set - establishing reference metrics)")
    print("="*60)
    
    ars_result = self.compute_attention_span_score(
        behavioral_data=behavioral_data,
        attention_metrics=attention_metrics,
        recovery_time=RecoveryLag,
        eeg_metric_name='theta_beta_ratio',
        weights=None,
        per_trial=False,
        baseline_metrics=baseline_metrics,
        is_baseline=is_baseline
    )
    
    # Handle return value based on whether this is baseline
    skipped_components = []
    if is_baseline and isinstance(ars_result, tuple):
        ass_value, raw_metrics = ars_result
        baseline_metrics_out = raw_metrics
    elif isinstance(ars_result, tuple):
        # Non-baseline: (aps_score, skipped_component_labels)
        ass_value, skipped_components = ars_result
        baseline_metrics_out = None
    else:
        ass_value = ars_result
        baseline_metrics_out = None

    # Optionally compute per-trial ARS (only for distraction sets)
    per_trial_ass = None
    if not is_baseline and per_trial_recovery is not None and len(per_trial_recovery) > 0:
        print("\nComputing per-trial ARS...")
        per_trial_ass = self.compute_attention_span_score(
            behavioral_data=behavioral_data,
            attention_metrics=attention_metrics,
            recovery_time=per_trial_recovery,
            eeg_metric_name='theta_beta_ratio',
            weights=None,
            per_trial=True,
            baseline_metrics=baseline_metrics
        )

    # Print final ARS
    print("\n" + "-"*60)
    if not np.isnan(ass_value) and ass_value != 0:
        if is_baseline:
            print(f"BASELINE ATTENTIONAL RESILIENCE SCORE: {ass_value:.1f}/100 (reference)")
            print("   This score represents 100% performance (no distractions)")
        else:
            # Display score - note it can exceed 100 if better than baseline
            if ass_value > 100:
                print(f"FINAL ATTENTIONAL RESILIENCE SCORE: {ass_value:.1f}% (ABOVE BASELINE!)")
            else:
                print(f"FINAL ATTENTIONAL RESILIENCE SCORE: {ass_value:.1f}%")
            
            if baseline_metrics:
                print(f"   (Relative to baseline performance)")
            
            # Interpretation relative to baseline - now handles >100%
            if ass_value > 110:
                interpretation = "Superior - Significantly BETTER than baseline (practice effect / heightened arousal)"
            elif ass_value > 100:
                interpretation = "Enhanced - Slightly better than baseline"
            elif ass_value >= 95:
                interpretation = "Excellent - Near-baseline performance despite distractions"
            elif ass_value >= 85:
                interpretation = "Good - Minor degradation from baseline"
            elif ass_value >= 70:
                interpretation = "Moderate - Some impact from distractions"
            elif ass_value >= 50:
                interpretation = "Below average - Significant distraction impact"
            else:
                interpretation = "Poor - Major distraction impact on performance"
            print(f"   Interpretation: {interpretation}")
    else:
        print(f"Could not calculate ARS - insufficient data")
    print("="*60)
    
    if per_trial_ass is not None:
        print(f"\nPer-trial ARS computed for {len(per_trial_ass)} trials")
        print(f"   Mean per-trial ARS: {per_trial_ass.mean():.3f}")
        print(f"   Sample (first 5 trials):\n{per_trial_ass.head()}")

    # 6. Print clean analysis summary
    self.print_analysis_summary(attention_metrics, behavioral_data, subject_id, level_id, task_id)
    
    # 7. Visualize results
    self.visualize_results(eeg_data, band_powers, attention_metrics, behavioral_data, level_id, task_id)
    
    return {
        'eeg_data': eeg_data,
        'band_powers': band_powers,
        'attention_metrics': attention_metrics,
        'behavioral_data': behavioral_data,
        'recovery_lag': RecoveryLag,
        'mean_recovery_per_trial': per_trial_recovery,
        'ARS': ass_value,
        'ARS_per_trial': per_trial_ass,
        'baseline_metrics': baseline_metrics_out,
        'skipped_aps_components': skipped_components,  # components excluded due to artifact
        'is_baseline': is_baseline
    }


## Run the interactive scorer

Run the next cell. It will prompt for level, subject, preprocessing, and ping times.


In [ ]:
def main():
"""Main analysis function for ALL LEVELS"""
print("="*70)
print("UNIFIED EEG ATTENTION ANALYSIS - ALL LEVELS")
print("Working with Muse headband data")
print("="*70)
print("\nThis analysis includes:")
print("  LEVEL 1: Classic, SART, Semantic")
print("  LEVEL 2: Selective Visual, Color-Number, Spatial Cue")
print("  LEVEL 3: Picture Recall, Color-Location, Dual Task")
print("="*70)

# Custom analyzer to handle various file naming conventions
class InteractiveAllLevelsAnalyzer(AllLevelsEEGAnalyzer):
    def load_behavioral_data(self, subject_id, task_name, set_id=None, level_id=1):
        '''Override to handle various file naming patterns with level-specific paths'''
        import os
        
        # Use level-specific path (not legacy csv_path)
        subject_dir = self.get_csv_path_for_level(level_id, subject_id)
        
        print(f"Looking for behavioral data in: {subject_dir}")
        
        if not os.path.exists(subject_dir):
            # Try alternate paths for this level
            level_base = self.get_csv_path_for_level(level_id)
            print(f'Subject directory not found: {subject_dir}')
            print(f'Level base path: {level_base}')
            
            # For Level 2, try flat structure
            if level_id == 2 and os.path.exists(level_base):
                subject_dir = level_base
                print(f'Using flat structure for Level 2: {subject_dir}')
            else:
                return None
            
        # Handle case variations in task names
        task_name_lower = task_name.lower()
        task_name_cap = task_name.capitalize()
        
        csv_files = os.listdir(subject_dir)
        
        # Try different patterns
        patterns = [
            f'{task_name}_results_set{set_id}_' if set_id else f'{task_name}_results_',
            f'{task_name_lower}_results_set{set_id}_' if set_id else f'{task_name_lower}_results_',
            f'{task_name_cap}_results_set{set_id}_' if set_id else f'{task_name_cap}_results_',
            f'{task_name}_results_',
            f'{task_name_lower}_results_'
        ]
        
        matching_files = []
        for pattern in patterns:
            matching = [f for f in csv_files if pattern in f and f.endswith('.csv')]
            if matching:
                matching_files.extend(matching)
                break
        
        if not matching_files:
            print(f'No behavioral data found for Subject {subject_id}, Level {level_id}, Task {task_name}')
            print(f'  Searched in: {subject_dir}')
            print(f'  Patterns tried: {patterns[:3]}')
            print(f'  Available CSV files: {[f for f in csv_files if f.endswith(".csv")][:5]}')
            return None
            
        csv_file = sorted(matching_files)[-1]
        filepath = os.path.join(subject_dir, csv_file)
        
        print(f'Loading behavioral data: {filepath}')
        return pd.read_csv(filepath)

analyzer = InteractiveAllLevelsAnalyzer()

# Step 1: Select which levels to analyze
print("\nSTEP 1: Level Selection")
print("-" * 30)
print("1. Level 1 only (Classic, SART, Semantic)")
print("2. Level 2 only (Selective Visual, Color-Number, Spatial Cue)")
print("3. Level 3 only (Picture Recall, Color-Location, Dual Task)")
print("4. All Levels (1, 2, and 3)")

while True:
    level_choice = input("Enter choice (1, 2, 3, or 4): ").strip()
    if level_choice in ['1', '2', '3', '4']:
        if level_choice == '4':
            selected_levels = [1, 2, 3]
        else:
            selected_levels = [int(level_choice)]
        print(f"Selected: Level(s) {selected_levels}")
        break
    else:
        print("Please enter 1, 2, 3, or 4.")

# Step 2: Get subject ID for each selected level
print("\nSTEP 2: Subject Selection")
print("-" * 30)

subject_ids_per_level = {}

for level_id in selected_levels:
    print(f"\nLevel {level_id} Subject Selection:")
    
    # Show available subjects for this level
    available_subjects = analyzer.find_available_subjects(level_id)
    
    if available_subjects:
        if 0 in available_subjects:
            # Level 2 flat structure
            print(f"   Level {level_id} has a flat CSV structure (no subject folders)")
            print(f"   Available data in: {analyzer.get_csv_path_for_level(level_id)}")
        else:
            print(f"   Available subjects for Level {level_id}: {available_subjects}")
    else:
        # Check directories directly
        csv_path = analyzer.get_csv_path_for_level(level_id)
        xdf_path = analyzer.get_xdf_path_for_level(level_id)
        print(f"   CSV path: {csv_path}")
        print(f"   XDF path: {xdf_path}")
        
        if os.path.exists(csv_path):
            try:
                contents = os.listdir(csv_path)
                subject_dirs = [d for d in contents if d.startswith('Subject') or d.startswith('subject')]
                if subject_dirs:
                    print(f"   Found: {subject_dirs}")
                else:
                    print(f"   Contents: {contents[:5]}..." if len(contents) > 5 else f"   Contents: {contents}")
            except:
                pass
    
    while True:
        try:
            subject_input = input(f"   Enter Subject ID for Level {level_id} (e.g., 2): ").strip()
            subject_id = int(subject_input)
            subject_ids_per_level[level_id] = subject_id
            
            # Verify data exists
            csv_dir = analyzer.get_csv_path_for_level(level_id, subject_id)
            if os.path.exists(csv_dir):
                print(f"   Found data at: {csv_dir}")
            else:
                # For level 2 flat structure, just use the base path
                base_csv = analyzer.get_csv_path_for_level(level_id)
                if os.path.exists(base_csv):
                    print(f"   Using flat structure: {base_csv}")
                else:
                    print(f"   Warning: Directory not found, but continuing...")
            break
            
        except ValueError:
            print("   Please enter a valid number.")

# Step 3: Get preprocessing choice
print("\nSTEP 3: Preprocessing Selection")  
print("-" * 30)
print("1. Basic preprocessing (fast, stable)")
print("2. Advanced preprocessing (research-grade, includes ICA)")

while True:
    choice = input("Enter choice (1 or 2): ").strip()
    if choice in ['1', '2']:
        advanced = (choice == '2')
        preprocessing_type = "Advanced" if advanced else "Basic"
        print(f"Selected: {preprocessing_type} preprocessing")
        break
    else:
        print("Please enter 1 or 2.")

# Step 4: Get stimulus times for each level, set, and task
print("\nSTEP 4: Stimulus Times Configuration")
print("-" * 40)
print("Configure stimulus times for each set and task:")
print("Set 1: Baseline (no distraction)")
print("Set 2: Distraction + Response required") 
print("Set 3: Distraction + No response required")
print()

def parse_time_to_seconds(time_str):
    """Convert time format like '1:30pm' to seconds from midnight"""
    time_str = time_str.strip().lower()
    
    if 'pm' in time_str or 'am' in time_str:
        time_str = time_str.replace(' ', '')
        try:
            time_obj = datetime.strptime(time_str, '%I:%M%p').time()
            total_seconds = time_obj.hour * 3600 + time_obj.minute * 60 + time_obj.second
            return total_seconds
        except ValueError:
            try:
                time_obj = datetime.strptime(time_str, '%I%p').time()
                total_seconds = time_obj.hour * 3600 + time_obj.minute * 60 + time_obj.second
                return total_seconds
            except ValueError:
                return None
    else:
        try:
            return float(time_str)
        except ValueError:
            return None

def convert_times_to_relative(time_list):
    """Convert absolute times to relative times from first stimulus"""
    if not time_list or len(time_list) == 0:
        return []
    
    sorted_times = sorted(time_list)
    baseline_time = sorted_times[0]
    
    relative_times = [(t - baseline_time) for t in sorted_times]
    return relative_times

stim_times_data = {}  # Key: (level_id, task_id, set_id)

for level_id in selected_levels:
    task_names = analyzer.task_names_by_level[level_id]
    
    print(f"\n{'='*50}")
    print(f"LEVEL {level_id} STIMULUS TIMES")
    print(f"{'='*50}")
    
    for set_id in [1, 2, 3]:
        set_condition = {
            1: "BASELINE (No Distraction)",
            2: "DISTRACTION + RESPONSE", 
            3: "DISTRACTION + NO RESPONSE"
        }[set_id]
        
        print(f"\nSET {set_id}: {set_condition}")
        print("-" * 25)
        
        for task_id in [1, 2, 3]:
            task_name = task_names[task_id]
            display_name = analyzer.get_display_name(task_name)
            
            print(f"\n{display_name} (Task {task_id}):")
            
            if set_id == 1:
                print("   Baseline set - no stimulus times needed")
                stim_times_data[(level_id, task_id, set_id)] = []
            else:
                print("   Enter stimulus times (formats supported):")
                print("     • Time format: 1:30pm, 1:31pm, 1:32pm")
                print("     • Or seconds: 0,60,120,180")
                print("   Press Enter for no stimuli")
                
                while True:
                    times_input = input(f"   Stimulus times for {display_name} Set {set_id}: ").strip()
                    
                    if times_input == "":
                        stim_times_data[(level_id, task_id, set_id)] = []
                        print("   No stimulus times")
                        break
                    else:
                        try:
                            time_strings = [t.strip() for t in times_input.split(',')]
                            parsed_times = []
                            
                            for time_str in time_strings:
                                parsed_time = parse_time_to_seconds(time_str)
                                if parsed_time is not None:
                                    parsed_times.append(parsed_time)
                                else:
                                    raise ValueError(f"Could not parse time: {time_str}")
                            
                            if parsed_times:
                                relative_times = convert_times_to_relative(parsed_times)
                                # Store BOTH original absolute times and relative times
                                stim_times_data[(level_id, task_id, set_id)] = {
                                    'absolute': parsed_times,  # Seconds from midnight
                                    'relative': relative_times  # Relative to first stimulus
                                }
                                
                                print(f"   Added {len(relative_times)} stimulus times:")
                                print(f"      Original: {time_strings}")
                                print(f"      Absolute (sec from midnight): {[f'{t:.0f}s' for t in parsed_times]}")
                                print(f"      Relative (from first stim): {[f'{t:.0f}s' for t in relative_times]}")
                                break
                            else:
                                raise ValueError("No valid times parsed")
                                
                        except ValueError as e:
                            print(f"   Invalid format: {e}")
                            print("   Examples: '1:30pm, 1:31pm, 1:32pm' or '0,60,120'")

# Step 5: Sync stimulus times with CSV files
print("\nSTEP 5: Syncing Stimulus Times with Actual Data Timestamps")
print("-" * 60)

def find_closest_timestamp_match(target_unix_time, xdf_timestamps, tolerance_seconds=30):
    """Find the closest timestamp match in XDF data within tolerance"""
    if len(xdf_timestamps) == 0:
        return None, None
        
    differences = np.abs(np.array(xdf_timestamps) - target_unix_time)
    min_diff_idx = np.argmin(differences)
    min_diff = differences[min_diff_idx]
    
    if min_diff <= tolerance_seconds:
        return min_diff_idx, min_diff
    else:
        return None, min_diff

def find_xdf_file(level_id, task_id, set_id, subject_id):
    """Find XDF file across possible paths for a specific level and subject"""
    
    # Get level-specific XDF paths - consistent structure: Level{N}/Results_xdf/Subject{M}/
    xdf_base_paths = [
        analyzer.get_xdf_path_for_level(level_id, subject_id),
        os.path.join(analyzer.base_path, f'Level{level_id}/Results_xdf/Subject{subject_id}'),
        os.path.join(analyzer.base_path, f'Level{level_id}/Results_xdf/subject{subject_id}'),  # legacy lowercase
    ]
    
    # Add legacy paths for backward compatibility
    xdf_base_paths.extend([
        os.path.join(analyzer.base_path, f'GoNoGoTests/Level{level_id}/Results_xdf/Subject{subject_id}'),
        os.path.join(analyzer.base_path, f'Results_xdf/Subject{subject_id}'),
        os.path.join(analyzer.base_path, f'Results_xdf/subject{subject_id}'),
    ])
    
    xdf_patterns = [
        f'sub-{{{subject_id}}}_task{{{task_id}}}_set{{{set_id}}}.xdf',
        f'sub-{subject_id}_task{task_id}_set{set_id}.xdf',
        f'sub-{{{subject_id}}}_level{{{level_id}}}_task{{{task_id}}}_set{{{set_id}}}.xdf'
    ]
    
    for xdf_base in xdf_base_paths:
        if not os.path.exists(xdf_base):
            continue
        for pattern in xdf_patterns:
            xdf_filepath = os.path.join(xdf_base, pattern)
            if os.path.exists(xdf_filepath):
                return xdf_filepath
        # Also check for any matching file
        try:
            for f in os.listdir(xdf_base):
                if f.endswith('.xdf') and (f'set{{{set_id}}}' in f or f'set{set_id}' in f):
                    return os.path.join(xdf_base, f)
        except:
            pass
    return None

def sync_with_xdf_timestamps(csv_filepath, xdf_filepath, stim_times_absolute, stim_times_relative, display_name, set_id):
    """
    Properly synchronize stimulus times with XDF and CSV timestamps.
    
    Args:
        csv_filepath: Path to behavioral CSV file
        xdf_filepath: Path to XDF/EEG file  
        stim_times_absolute: List of stimulus times in seconds from midnight
        stim_times_relative: List of stimulus times relative to first stimulus
        display_name: Task display name
        set_id: Set number (1, 2, or 3)
    """
    try:
        df_csv = pd.read_csv(csv_filepath)
        df_csv['stim_time'] = np.nan
        
        # Use absolute times if available, otherwise relative
        stim_times_to_use = stim_times_absolute if stim_times_absolute else stim_times_relative
        use_absolute = bool(stim_times_absolute)
        
        if len(stim_times_to_use) == 0:
            print(f'   No stimulus times (baseline set)')
            df_csv.to_csv(csv_filepath, index=False)
            return True
        
        # Check if CSV has timestamp column
        csv_has_timestamps = 'timestamp' in df_csv.columns
        
        # Try to load XDF data (it may be converted to CSV format already)
        xdf_timestamps = None
        try:
            # First try reading as CSV (if already converted)
            df_xdf = pd.read_csv(xdf_filepath)
            if 'timestamps' in df_xdf.columns:
                xdf_timestamps = df_xdf['timestamps'].values
                print(f'   XDF (CSV format): {len(xdf_timestamps)} samples')
        except:
            pass
        
        if xdf_timestamps is None:
            # Try loading as actual XDF file using pyxdf
            try:
                import pyxdf
                streams, _ = pyxdf.load_xdf(xdf_filepath)
                # Find the EEG stream
                for stream in streams:
                    if len(stream['time_stamps']) > 0:
                        xdf_timestamps = stream['time_stamps']
                        print(f'   XDF (binary): {len(xdf_timestamps)} samples')
                        break
            except ImportError:
                print(f'   pyxdf not installed - using basic sync')
            except Exception as e:
                print(f'   Could not load XDF: {e}')
        
        if xdf_timestamps is not None and len(xdf_timestamps) > 0:
            # PROPER SYNCHRONIZATION
            first_xdf_time = xdf_timestamps[0]
            last_xdf_time = xdf_timestamps[-1]
            xdf_duration = last_xdf_time - first_xdf_time
            
            print(f'   XDF recording duration: {xdf_duration:.1f}s')
            print(f'   XDF start time (Unix): {first_xdf_time:.2f}')
            
            # If CSV has timestamps, use them for better sync
            if csv_has_timestamps:
                try:
                    # Parse CSV timestamps (they are in local time)
                    csv_first_timestamp = pd.to_datetime(df_csv['timestamp'].iloc[0])
                    csv_last_timestamp = pd.to_datetime(df_csv['timestamp'].iloc[-1])
                    csv_duration = (csv_last_timestamp - csv_first_timestamp).total_seconds()
                    
                    # The task duration in CSV and XDF should be similar
                    # Use this to find the offset rather than raw Unix timestamp comparison
                    # XDF starts before CSV, so the CSV starts at some point during XDF recording
                    
                    # Estimate when CSV started relative to XDF start
                    # If XDF is 300s and CSV is 100s, CSV probably started around XDF second 200
                    if xdf_duration > 0 and csv_duration > 0:
                        # Assume CSV starts when task data actually began
                        # The offset should be roughly: xdf_duration - csv_duration
                        estimated_offset = max(0, xdf_duration - csv_duration)
                        print(f'   CSV task duration: {csv_duration:.1f}s')
                        print(f'   Estimated CSV start: {estimated_offset:.1f}s into XDF recording')
                        offset = estimated_offset
                    else:
                        offset = 0
                except Exception as e:
                    print(f'   Could not parse CSV timestamps: {e}')
                    offset = 0
            else:
                offset = 0
            
            print(f'   Syncing {len(stim_times_to_use)} stimuli ({"absolute" if use_absolute else "relative"} times)...')
            
            matched_stimuli = 0
            
            if csv_has_timestamps and use_absolute:
                # BEST APPROACH: Match absolute times (seconds from midnight) directly with CSV timestamps
                # Parse all CSV timestamps to seconds from midnight for comparison
                csv_times_from_midnight = []
                for ts in df_csv['timestamp']:
                    try:
                        dt = pd.to_datetime(ts)
                        seconds_from_midnight = dt.hour * 3600 + dt.minute * 60 + dt.second + dt.microsecond / 1e6
                        csv_times_from_midnight.append(seconds_from_midnight)
                    except:
                        csv_times_from_midnight.append(np.nan)
                
                csv_times_array = np.array(csv_times_from_midnight)
                
                # Pre-process stimulus times to spread out duplicates
                # If multiple stimuli have the same minute, spread them out
                processed_stim_times = list(stim_times_absolute)
                
                # Group by minute and spread duplicates
                from collections import Counter
                minute_counts = Counter([int(t // 60) for t in processed_stim_times])
                minute_indices = {}
                
                for i, abs_time in enumerate(processed_stim_times):
                    minute = int(abs_time // 60)
                    if minute_counts[minute] > 1:
                        # Multiple stimuli in this minute - spread them out
                        if minute not in minute_indices:
                            minute_indices[minute] = 0
                        idx_in_minute = minute_indices[minute]
                        n_in_minute = minute_counts[minute]
                        # Spread evenly within the minute (0-59 seconds)
                        offset = (idx_in_minute * 60) / n_in_minute
                        processed_stim_times[i] = minute * 60 + offset
                        minute_indices[minute] += 1
                
                # Track already matched trial indices to ensure unique matches
                matched_trial_indices = set()
                
                for i, abs_time in enumerate(processed_stim_times):
                    # abs_time is in seconds from midnight (e.g., 10:55am = 39300)
                    # Find the trial that occurred closest to this time (excluding already matched)
                    
                    differences = np.abs(csv_times_array - abs_time)
                    # Mask out already matched trials
                    for matched_idx in matched_trial_indices:
                        differences[matched_idx] = np.inf
                    
                    min_idx = np.nanargmin(differences)
                    min_diff = differences[min_idx]
                    
                    if min_diff <= 120:  # Within 2 minute tolerance (increased for spread times)
                        matched_stimuli += 1
                        matched_trial_indices.add(min_idx)
                        # Store the relative time for consistency
                        relative_time = stim_times_relative[i] if i < len(stim_times_relative) else 0
                        df_csv.loc[min_idx, 'stim_time'] = relative_time
                        
                        trial_time = datetime.strptime(str(df_csv.loc[min_idx, 'timestamp'])[:19], '%Y-%m-%d %H:%M:%S')
                        orig_time = stim_times_absolute[i]
                        stim_time_str = f"{int(orig_time//3600)}:{int((orig_time%3600)//60):02d}"
                        print(f'     Stimulus {i+1}: {stim_time_str} → Trial {min_idx} at {trial_time.strftime("%I:%M:%S %p")} (±{min_diff:.1f}s)')
                    else:
                        orig_time = stim_times_absolute[i]
                        stim_time_str = f"{int(orig_time//3600)}:{int((orig_time%3600)//60):02d}"
                        print(f'     Stimulus {i+1}: {stim_time_str} → No close match (closest: {min_diff:.1f}s)')
            
            elif csv_has_timestamps:
                # Use relative times with CSV timestamps
                csv_times_from_midnight = []
                for ts in df_csv['timestamp']:
                    try:
                        dt = pd.to_datetime(ts)
                        seconds_from_midnight = dt.hour * 3600 + dt.minute * 60 + dt.second + dt.microsecond / 1e6
                        csv_times_from_midnight.append(seconds_from_midnight)
                    except:
                        csv_times_from_midnight.append(np.nan)
                
                csv_times_array = np.array(csv_times_from_midnight)
                first_csv_time = csv_times_array[0] if len(csv_times_array) > 0 else 0
                
                # Track already matched trial indices to ensure unique matches
                matched_trial_indices = set()
                
                for i, relative_time in enumerate(stim_times_relative):
                    target_time = first_csv_time + relative_time
                    
                    differences = np.abs(csv_times_array - target_time)
                    # Mask out already matched trials
                    for matched_idx in matched_trial_indices:
                        differences[matched_idx] = np.inf
                    
                    min_idx = np.nanargmin(differences)
                    min_diff = differences[min_idx]
                    
                    if min_diff <= 300:  # Within 5 minutes tolerance
                        matched_stimuli += 1
                        matched_trial_indices.add(min_idx)
                        df_csv.loc[min_idx, 'stim_time'] = relative_time
                        
                        trial_time = datetime.strptime(str(df_csv.loc[min_idx, 'timestamp'])[:19], '%Y-%m-%d %H:%M:%S')
                        print(f'     Stimulus {i+1}: +{relative_time:.0f}s → Trial {min_idx} at {trial_time.strftime("%I:%M:%S %p")}')
                    else:
                        print(f'     Stimulus {i+1}: +{relative_time:.0f}s → No close match (closest: {min_diff:.1f}s)')
            else:
                # Fall back to XDF-based matching
                # Track already matched trial indices to ensure unique matches
                matched_trial_indices = set()
                
                for i, relative_time in enumerate(stim_times_relative):
                    absolute_stimulus_time = first_xdf_time + offset + relative_time
                    
                    differences = np.abs(xdf_timestamps - absolute_stimulus_time)
                    min_diff_idx = np.argmin(differences)
                    min_diff = differences[min_diff_idx]
                    
                    if min_diff <= 300:  # Within 5 minutes tolerance
                        samples_per_trial = len(xdf_timestamps) / len(df_csv)
                        estimated_trial_idx = int(min_diff_idx / samples_per_trial)
                        estimated_trial_idx = max(0, min(estimated_trial_idx, len(df_csv) - 1))
                        
                        # Ensure unique match - find next available trial if this one is taken
                        while estimated_trial_idx in matched_trial_indices and estimated_trial_idx < len(df_csv) - 1:
                            estimated_trial_idx += 1
                        
                        if estimated_trial_idx not in matched_trial_indices:
                            matched_stimuli += 1
                            matched_trial_indices.add(estimated_trial_idx)
                            df_csv.loc[estimated_trial_idx, 'stim_time'] = relative_time
                            print(f'     Stimulus {i+1}: +{relative_time:.0f}s → Trial {estimated_trial_idx}')
                        else:
                            print(f'     Stimulus {i+1}: No available unique trial')
                    else:
                        print(f'     Stimulus {i+1}: No match (closest: {min_diff:.1f}s)')
            
            print(f'   Successfully matched {matched_stimuli}/{len(stim_times_to_use)} stimuli')
            
        else:
            # FALLBACK: Basic sync without XDF
            print(f'   Using basic sync (no XDF timestamps available)')
            n_trials = len(df_csv)
            stim_times_for_fallback = stim_times_relative if stim_times_relative else stim_times_absolute
            indices = np.linspace(0, n_trials-1, len(stim_times_for_fallback), dtype=int)
            for i, stim_time in enumerate(stim_times_for_fallback):
                if i < len(indices):
                    df_csv.loc[indices[i], 'stim_time'] = stim_time
            print(f'   Distributed {len(stim_times_for_fallback)} stimulus times across {n_trials} trials')
        
        df_csv.to_csv(csv_filepath, index=False)
        print(f'   Updated CSV with stim_time column')
        return True
        
    except Exception as e:
        print(f'   Error in timestamp sync: {e}')
        return False

# Process each level, task, and set
for level_id in selected_levels:
    task_names = analyzer.task_names_by_level[level_id]
    subject_id = subject_ids_per_level[level_id]
    
    # Get level-specific CSV path
    csv_base_path = analyzer.get_csv_path_for_level(level_id, subject_id)
    
    # Also try alternative paths for this level
    csv_paths_to_try = [
        csv_base_path,
        os.path.join(analyzer.base_path, f"Level{level_id}/Results_CSV/Subject{subject_id}"),
        os.path.join(analyzer.base_path, f"Level{level_id}/Results_CSV/Level{level_id}"),
        os.path.join(analyzer.base_path, f"Level{level_id}/Results_CSV"),
    ]
    
    print(f"\n{'='*50}")
    print(f"Processing Level {level_id} for Subject {subject_id}")
    print(f"CSV search paths: {csv_paths_to_try}")
    print(f"{'='*50}")
    
    for task_id in [1, 2, 3]:
        task_name = task_names[task_id]
        
        for set_id in [1, 2, 3]:
            # Find actual CSV filename across possible paths
            csv_patterns = [
                f'{task_name}_results_set{set_id}_',
                f'{task_name.lower()}_results_set{set_id}_',
                f'{task_name.capitalize()}_results_set{set_id}_'
            ]
            
            files = []
            found_base_path = None
            for base_path in csv_paths_to_try:
                if not os.path.exists(base_path):
                    continue
                for pattern in csv_patterns:
                    matching = [f for f in os.listdir(base_path) if pattern in f and f.endswith('.csv')]
                    if matching:
                        files = matching
                        found_base_path = base_path
                        break
                if files:
                    break
            
            if files and found_base_path:
                filename = sorted(files)[-1]  # Use most recent
                filepath = os.path.join(found_base_path, filename)
                
                display_name = analyzer.get_display_name(task_name)
                print(f'\nProcessing Level {level_id}, Task {task_id} ({display_name}), Set {set_id}')
                print(f'   CSV: {filename}')
                
                # Find corresponding XDF file
                xdf_filepath = find_xdf_file(level_id, task_id, set_id, subject_id)
                if xdf_filepath:
                    print(f'   XDF: {os.path.basename(xdf_filepath)}')
                else:
                    print(f'   XDF: Not found')
                
                # Get stimulus times for this session
                stim_times_entry = stim_times_data.get((level_id, task_id, set_id), {})
                
                # Handle both old format (list) and new format (dict)
                if isinstance(stim_times_entry, dict):
                    stim_times_absolute = stim_times_entry.get('absolute', [])
                    stim_times_relative = stim_times_entry.get('relative', [])
                else:
                    # Old format - list of relative times
                    stim_times_absolute = []
                    stim_times_relative = stim_times_entry if stim_times_entry else []
                
                # Synchronize timestamps
                if xdf_filepath:
                    sync_with_xdf_timestamps(filepath, xdf_filepath, stim_times_absolute, stim_times_relative, display_name, set_id)
                else:
                    # Fallback without XDF
                    try:
                        df = pd.read_csv(filepath)
                        df['stim_time'] = np.nan
                        stim_times = stim_times_relative if stim_times_relative else stim_times_absolute
                        if len(stim_times) > 0:
                            n_trials = len(df)
                            indices = np.linspace(0, n_trials-1, len(stim_times), dtype=int)
                            for i, stim_time in enumerate(stim_times):
                                if i < len(indices):
                                    df.loc[indices[i], 'stim_time'] = stim_time
                            print(f'   Added {len(stim_times)} stimulus times (basic sync)')
                        else:
                            print(f'   No stimulus times (baseline set)')
                        df.to_csv(filepath, index=False)
                    except Exception as e:
                        print(f'   Error: {e}')

# Step 6: Run complete analysis
print("\nSTEP 6: Running Complete Analysis")
print("-" * 40)

all_sessions = []

for level_id in selected_levels:
    task_names = analyzer.task_names_by_level[level_id]
    subject_id = subject_ids_per_level[level_id]  # Get subject ID for this level
    
    for set_id in [1, 2, 3]:
        set_condition = {
            1: 'Baseline',
            2: 'Distraction/Response', 
            3: 'Distraction/NoResponse'
        }[set_id]
        
        for task_id in [1, 2, 3]:
            task_name = task_names[task_id]
            display_name = analyzer.get_display_name(task_name)
            session_name = f'L{level_id}_S{subject_id}_{display_name} Set {set_id} - {set_condition}'
            
            all_sessions.append((subject_id, level_id, task_id, set_id, session_name))

results_by_level_set = {}  # (level_id, set_id) -> {task_id -> results}
all_results = {}

# Caches keyed by (level_id, task_id) — all populated after Set 1 runs
baseline_metrics_store    = {}   # -> baseline scalar metrics dict
baseline_eeg_store        = {}   # -> preprocessed Set 1 EEG DataFrame
baseline_behavioral_store = {}   # -> Set 1 behavioral DataFrame

for subj_id, level_id, task_id, set_id, session_name in all_sessions:
    print(f'\nAnalyzing {session_name}...')

    baseline_key = (level_id, task_id)
    baseline_metrics          = baseline_metrics_store.get(baseline_key)    if set_id > 1 else None
    baseline_eeg_data         = baseline_eeg_store.get(baseline_key)        if set_id > 1 else None
    baseline_behavioral_data  = baseline_behavioral_store.get(baseline_key) if set_id > 1 else None

    if set_id > 1 and baseline_metrics:
        print(f"   Using baseline metrics from Set 1 for normalization")
    elif set_id > 1:
        print(f"   Warning: No baseline metrics available (Set 1 not processed first)")

    if set_id > 1 and baseline_eeg_data is not None and baseline_behavioral_data is not None:
        print(f"   V3: Set 1 EEG + behavioral data ready for trial-aligned recovery")
    elif set_id > 1:
        print(f"   V3: Set 1 data not cached yet — recovery will fall back to local baseline")

    try:
        results = analyzer.analyze_subject(
            subject_id=subj_id,
            level_id=level_id,
            task_id=task_id,
            set_id=set_id,
            advanced_preprocessing=advanced,
            baseline_metrics=baseline_metrics,
            baseline_eeg_data=baseline_eeg_data,
            baseline_behavioral_data=baseline_behavioral_data
        )

        if results:
            all_results[session_name] = results

            if set_id == 1:
                # Cache everything from Set 1 for Sets 2 & 3
                if results.get('baseline_metrics'):
                    baseline_metrics_store[baseline_key] = results['baseline_metrics']
                    print(f"   Cached baseline metrics for Level {level_id}, Task {task_id}")
                if results.get('eeg_data') is not None:
                    baseline_eeg_store[baseline_key] = results['eeg_data']
                    print(f"   Cached Set 1 EEG for Level {level_id}, Task {task_id} (V3 recovery)")
                if results.get('behavioral_data') is not None:
                    baseline_behavioral_store[baseline_key] = results['behavioral_data']
                    print(f"   Cached Set 1 behavioral data for Level {level_id}, Task {task_id} (V3 recovery)")
            
            key = (level_id, set_id)
            if key not in results_by_level_set:
                results_by_level_set[key] = {}
            results_by_level_set[key][task_id] = results
            
            print(f'SUCCESS: {session_name}')
        else:
            print(f'FAILED: {session_name} - No results')
            
    except Exception as e:
        print(f'ERROR: {session_name} - {str(e)[:60]}...')

# Step 7: Generate comprehensive tabular results
print("\nSTEP 7: Comprehensive Results Analysis")
print("="*80)

generate_all_levels_table(subject_ids_per_level, selected_levels, results_by_level_set, all_results, analyzer)

def generate_all_levels_table(subject_ids_per_level, selected_levels, results_by_level_set, all_results, analyzer):
"""Generate comprehensive tabular analysis for all levels"""

# Build subject info string
subject_info = ", ".join([f"L{lvl}:S{sid}" for lvl, sid in subject_ids_per_level.items()])
print(f"\nCOMPREHENSIVE ATTENTION ANALYSIS - ALL LEVELS")
print(f"Subjects: {subject_info}")
print("="*100)

table_data = []

set_conditions = {
    1: 'Baseline',
    2: 'Distraction+Response', 
    3: 'Distraction+NoResponse'
}

for level_id in selected_levels:
    task_names = analyzer.task_names_by_level[level_id]
    
    for set_id in [1, 2, 3]:
        for task_id in [1, 2, 3]:
            key = (level_id, set_id)
            
            if key in results_by_level_set and task_id in results_by_level_set[key]:
                results = results_by_level_set[key][task_id]
                
                attention_metrics = results.get('attention_metrics', {})
                behavioral_data = results.get('behavioral_data', None)
                ass_score = results.get('ARS', np.nan)
                
                # Extract metrics (handle different column names)
                accuracy = np.nan
                if behavioral_data is not None:
                    for col in ['response_correct', 'is_correct', 'correct', 'accuracy']:
                        if col in behavioral_data.columns:
                            accuracy = behavioral_data[col].mean()
                            break
                
                avg_rt = np.nan
                rt_std = np.nan
                if behavioral_data is not None:
                    for col in ['rt', 'reaction_time']:
                        if col in behavioral_data.columns:
                            avg_rt = behavioral_data[col].mean()
                            rt_std = behavioral_data[col].std()
                            break
                
                _DISP_EEG_MIN = 0.01  # exclude dead-electrode zeros from display average
                tb_ratio = np.nan
                if 'theta_beta_ratio' in attention_metrics:
                    tb_values = list(attention_metrics['theta_beta_ratio'].values())
                    tb_ratio = np.mean([v for v in tb_values if not np.isnan(v) and v > _DISP_EEG_MIN]) if tb_values else np.nan
                
                ab_ratio = np.nan
                if 'alpha_beta_ratio' in attention_metrics:
                    ab_values = list(attention_metrics['alpha_beta_ratio'].values())
                    ab_ratio = np.mean([v for v in ab_values if not np.isnan(v) and v > _DISP_EEG_MIN]) if ab_values else np.nan
                
                recovery_time = attention_metrics.get('mean_recovery_time', np.nan)
                frontal_asymmetry = attention_metrics.get('frontal_asymmetry', np.nan)
                trials_to_3_correct = attention_metrics.get('trials_to_3_consecutive_correct', np.nan)
                
                task_name = task_names[task_id]
                display_name = analyzer.get_display_name(task_name)
                
                skipped = results.get('skipped_aps_components', [])
                table_data.append({
                    'Level': level_id,
                    'Set': set_id,
                    'Condition': set_conditions[set_id],
                    'Task': display_name,
                    'ARS_Score': ass_score,
                    'Accuracy_%': accuracy * 100 if not np.isnan(accuracy) else np.nan,
                    'RT_Mean_s': avg_rt,
                    'RT_Std_s': rt_std,
                    'Theta_Beta': tb_ratio,
                    'Alpha_Beta': ab_ratio,
                    'Recovery_s': recovery_time,
                    'Trials_to_3CC': trials_to_3_correct,
                    'Frontal_Asym': frontal_asymmetry,
                    # Show which specific EEG component(s) were skipped, not just Yes/No.
                    # Empty string means all EEG components were valid for this session.
                    'EEG_Skipped': ', '.join(skipped) if skipped else ''
                })

if len(table_data) == 0:
    print("No results to display.")
    return

df = pd.DataFrame(table_data)

# For non-baseline (distraction/recovery) rows, show 0 instead of NaN so
# the table is easier to read.  NaN is preserved for Set-1 baseline rows
# because there a NaN genuinely means "no signal was ever recorded."
_zero_fill_cols = ['Theta_Beta', 'Alpha_Beta', 'Frontal_Asym', 'Recovery_s', 'Trials_to_3CC']
df_display = df.copy()
for _col in _zero_fill_cols:
    df_display.loc[df_display['Set'] > 1, _col] = (
        df_display.loc[df_display['Set'] > 1, _col].fillna(0)
    )

print("\nDETAILED RESULTS TABLE:")
print("-" * 140)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.2f}'.format)

display_cols = ['Level', 'Set', 'Condition', 'Task', 'ARS_Score', 'Accuracy_%',
                'RT_Mean_s', 'RT_Std_s', 'Theta_Beta', 'Alpha_Beta',
                'Recovery_s', 'Trials_to_3CC', 'Frontal_Asym', 'EEG_Skipped']
print(df_display[display_cols].to_string(index=False, float_format='{:.2f}'.format))

# Summary by Level
print("\n\nCROSS-LEVEL COMPARISON:")
print("="*60)

for level_id in selected_levels:
    level_df = df[df['Level'] == level_id]
    if len(level_df) == 0:
        continue
        
    print(f"\nLEVEL {level_id} SUMMARY:")
    print("-" * 40)
    
    for set_id in [1, 2, 3]:
        set_df = level_df[level_df['Set'] == set_id]
        if len(set_df) == 0:
            continue
            
        condition = set_conditions[set_id]
        
        ass_mean = set_df['ARS_Score'].mean()
        ass_std = set_df['ARS_Score'].std()
        acc_mean = set_df['Accuracy_%'].mean()
        rt_mean = set_df['RT_Mean_s'].mean()
        
        print(f"  Set {set_id} ({condition}):")
        print(f"    ARS: {ass_mean:.1f} ± {ass_std:.1f}")
        print(f"    Accuracy: {acc_mean:.1f}%")
        print(f"    RT: {rt_mean:.3f}s")

# Best/Worst across all levels
print("\n\nPERFORMANCE HIGHLIGHTS:")
print("="*40)

if not df['ARS_Score'].isna().all():
    best_ass = df.loc[df['ARS_Score'].idxmax()]
    worst_ass = df.loc[df['ARS_Score'].idxmin()]
    
    print(f"BEST:  Level {best_ass['Level']} - {best_ass['Task']} ({best_ass['Condition']}) - ARS: {best_ass['ARS_Score']:.1f}")
    print(f"WORST: Level {worst_ass['Level']} - {worst_ass['Task']} ({worst_ass['Condition']}) - ARS: {worst_ass['ARS_Score']:.1f}")

# Level comparison
if len(selected_levels) > 1:
    print("\n\nLEVEL COMPARISON (Average ARS by Difficulty):")
    print("-" * 40)
    
    for level_id in selected_levels:
        level_df = df[df['Level'] == level_id]
        if len(level_df) > 0:
            level_ass = level_df['ARS_Score'].mean()
            level_acc = level_df['Accuracy_%'].mean()
            print(f"  Level {level_id}: ARS = {level_ass:.1f}, Accuracy = {level_acc:.1f}%")

subject_info = ", ".join([f"L{lvl}:S{sid}" for lvl, sid in subject_ids_per_level.items()])
print(f"\nCOMPLETE ANALYSIS FINISHED!")
print(f"Subjects analyzed: {subject_info}")
print("="*60)

# Save results to file
# Get the primary subject ID (use first one if multiple levels have different subjects)
primary_subject_id = list(subject_ids_per_level.values())[0]
output_filename = f"Analysis_Subject{primary_subject_id}_trialMappedRecovery.txt"
output_path = os.path.join(analyzer.base_path, output_filename)

try:
    with open(output_path, 'w') as f:
        f.write("="*100 + "\n")
        f.write(f"COMPREHENSIVE ATTENTION ANALYSIS - SUBJECT {primary_subject_id}\n")
        f.write(f"Subjects analyzed: {subject_info}\n")
        f.write(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("="*100 + "\n\n")
        
        # Write the results table (hide internal _skipped_detail column)
        if len(table_data) > 0:
            df = pd.DataFrame(table_data)
            # Same NaN→0 substitution for non-baseline rows in the saved file
            _zero_fill_cols = ['Theta_Beta', 'Alpha_Beta', 'Frontal_Asym', 'Recovery_s', 'Trials_to_3CC']
            for _col in _zero_fill_cols:
                df.loc[df['Set'] > 1, _col] = df.loc[df['Set'] > 1, _col].fillna(0)
            file_display_cols = ['Level', 'Set', 'Condition', 'Task', 'ARS_Score',
                                 'Accuracy_%', 'RT_Mean_s', 'RT_Std_s', 'Theta_Beta',
                                 'Alpha_Beta', 'Recovery_s', 'Trials_to_3CC',
                                 'Frontal_Asym', 'EEG_Skipped']
            f.write("DETAILED RESULTS TABLE:\n")
            f.write("-"*120 + "\n")
            f.write(df[file_display_cols].to_string(index=False) + "\n")
            f.write("-"*120 + "\n\n")
            
            # Write summary statistics
            f.write("SUMMARY BY LEVEL AND SET:\n")
            f.write("-"*60 + "\n")
            for level_id in selected_levels:
                level_df = df[df['Level'] == level_id]
                if len(level_df) > 0:
                    f.write(f"\nLevel {level_id}:\n")
                    for set_id in [1, 2, 3]:
                        set_df = level_df[level_df['Set'] == set_id]
                        if len(set_df) > 0:
                            avg_ars = set_df['ARS_Score'].mean()
                            avg_acc = set_df['Accuracy_%'].mean()
                            f.write(f"  Set {set_id}: ARS = {avg_ars:.1f}, Accuracy = {avg_acc:.1f}%\n")
            
            # Data quality notes — list any sessions where APS components were skipped.
            # The EEG_Skipped column now contains the component name(s) directly,
            # e.g. "FAA" or "TBR, ABR, FAA". Empty string means no skips.
            quality_rows = df[df['EEG_Skipped'] != '']
            if len(quality_rows) > 0:
                f.write("\nDATA QUALITY NOTES:\n")
                f.write("-"*60 + "\n")
                f.write("The following sessions had one or more EEG components excluded\n")
                f.write("from APS scoring. A component is skipped when either the current\n")
                f.write("session OR the baseline session had no valid EEG signal — without\n")
                f.write("a baseline reference, normalisation is impossible.\n\n")
                f.write("FAA methodology note:\n")
                f.write("  Frontal Alpha Asymmetry is computed as the log-power difference:\n")
                f.write("    FAA = ln(AF8_alpha_power) - ln(AF7_alpha_power)\n")
                f.write("  where alpha_power = RMS^2 of the bandpass-filtered (8-13 Hz) signal.\n")
                f.write("  The valid physiological range is -1.0 to +1.0 (a value of ±1 means\n")
                f.write("  one hemisphere has ~2.72x the alpha power of the other). Values\n")
                f.write("  outside this range are treated as artifact and excluded.\n\n")
                f.write("How to read the table:\n")
                f.write("  Frontal_Asym = 0.00 in Set 2/3 row → FAA was NaN (invalid) for that\n")
                f.write("                                         session; 0.00 is display-only fill.\n")
                f.write("  Reason for FAA = NaN:\n")
                f.write("    (a) Dead/disconnected electrode — AF7 or AF8 RMS alpha power < 1e-4.\n")
                f.write("    (b) Out-of-range artifact     — computed log-FAA outside [-1.0, +1.0].\n")
                f.write("    (c) Baseline unavailable      — Set 1 FAA was also NaN, so\n")
                f.write("                                     normalisation against baseline is\n")
                f.write("                                     impossible.\n")
                f.write("  NaN in baseline (Set 1) row    → EEG was completely absent or\n")
                f.write("                                     unreadable for that session.\n\n")
                for _, row in quality_rows.iterrows():
                    f.write(f"  Level {int(row['Level'])} Set {int(row['Set'])} "
                            f"{row['Task']} ({row['Condition']}):\n")
                    f.write(f"    Skipped: {row['EEG_Skipped']}\n")
                    f.write(f"    APS calculated from remaining valid components only.\n\n")
                f.write("-"*60 + "\n")

            f.write("\n" + "="*100 + "\n")
            f.write(f"ANALYSIS COMPLETE\n")
            f.write("="*100 + "\n")
        
    print(f"\n*** Results saved to: {output_path} ***")
except Exception as e:
    print(f"\nWarning: Could not save results to file: {e}")

# Run this cell to execute the script
main()
